# 🧩 13 · Diagonalization and the singular value decomposition
### Similar matrices, diagonalization $A = X\Lambda X^{-1}$ and cheap matrix powers; the singular value decomposition $A = U\Sigma V^\top$ — its shapes, its geometry (circle → ellipse), its construction from $A^\top A$, low-rank approximation — and the eigengenes of an infection experiment

**BiologyToMath** · Part C — Linear algebra · Module 13 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/13_diagonalization_svd.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy (`np.linalg.eig`, `np.linalg.svd`), SymPy (exact diagonalization and SVD by hand), Matplotlib (including 3-D axes) |
| **Prerequisites** | [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) (eigenvalues, eigenvectors, the spectral theorem, $B^\top B \succeq 0$, power iteration). Modules 08–11: matrix products, transposes and determinants; inverses; orthonormal bases, change of basis and rank. |
| **Estimated time** | 160 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Equivalent matrices ($A' = T^{-1}AS$) and similar matrices ($B' = P^{-1}BP$); diagonal matrices; diagonalizable matrices; $AP = PD$ iff the columns of $P$ are eigenvectors and $D$ holds the eigenvalues; eigendecomposition $A = X\Lambda X^{-1}$ when the eigenvectors are linearly independent; the worked eigendecompositions and the symmetric case $A = PDP^\top$ (rotation, stretch, rotation back); the singular value decomposition — the theorem ($U$, $V$ orthogonal, $\Sigma$ padded diagonal with $\sigma_1\ge\sigma_2\ge\dots\ge0$, left and right singular vectors), the shapes for $n > m$ and $m > n$ and the thin form, the intuition of a map from $\mathbb R^2$ to $\mathbb R^3$, constructing the SVD from $A^\top A$ ($\mathbf u_i = A\mathbf v_i/\sigma_i$) and the worked example; applications to users × movies ratings and genes × assays expression data |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The expression data come from the course's **SIMULATED** infection experiment (`simulate_tissue_expression`: spleen, kidney, liver × control/infected × 4 replicates, 400 genes). The movie ratings, the population matrices and the drug × cell-line responses are **illustrative** (typed in, or simulated with a known low-rank truth and labelled SIMULATED).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Define** equivalent and similar matrices, **prove** that similar matrices share their characteristic polynomial
   (eigenvalues, determinant, trace), and **explain** why similarity is stricter than equivalence.
2. **Prove** column by column that $AP = PD$ exactly when the columns of $P$ are eigenvectors and $D$ holds the
   eigenvalues, and **decide** whether a matrix is diagonalizable (it needs $n$ independent eigen**vectors**).
3. **Compute** eigendecompositions $A = X\Lambda X^{-1}$ by hand (2 × 2 and 3 × 3), **check** them with SymPy and NumPy,
   and **recognise** non-diagonalizable matrices (the shear, a population matrix with a repeated eigenvalue).
4. **Read** $A = PDP^{-1}$ as "change to eigen-coordinates, stretch, change back" and **use** $A^k = X\Lambda^kX^{-1}$
   to project a population or a Markov chain many steps ahead in closed form.
5. **State** the SVD theorem $A = U\Sigma V^\top$ with all shapes (full and thin), **translate** between notations, and
   **avoid** the `Vh` trap of `np.linalg.svd`.
6. **Visualise** the SVD as rotate–stretch–rotate, mapping the unit circle to an ellipse with semi-axes $\sigma_i\mathbf u_i$.
7. **Construct** an SVD by hand from the eigendecomposition of $A^\top A$ ($\sigma_i = \sqrt{\lambda_i}$,
   $\mathbf u_i = A\mathbf v_i/\sigma_i$) and **relate** it to the eigendecomposition of symmetric PSD matrices.
8. **Apply** low-rank approximation (Eckart–Young) to denoise data, predict a missing rating or drug response, and
   **interpret** the singular vectors of an expression matrix as eigengenes that separate tissues and infection status.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the 24 × 400 infection expression matrix | 5 min |
| 1 · Equivalent and similar matrices | definitions, shared eigenvalues, a counterexample | 15 min |
| 2 · Diagonalization | $AP = PD$ column by column, $A = X\Lambda X^{-1}$, when it is possible | 15 min |
| 3 · Worked eigendecompositions | 2 × 2 and 3 × 3 by hand; matrices that cannot be diagonalized | 20 min |
| 4 · The geometry | change to eigen-coordinates, stretch, change back; symmetric = rotate–stretch–rotate | 10 min |
| 5 · Matrix powers | $A^k = X\Lambda^kX^{-1}$; a population in closed form; Markov chains | 15 min |
| 6 · The SVD theorem | shapes, padding, thin form, notations, `np.linalg.svd` | 15 min |
| 7 · SVD intuition | a map from $\mathbb R^2$ to $\mathbb R^3$: rotate, stretch, rotate | 10 min |
| 8 · Constructing the SVD | from $A^\top A$ by hand: the worked example | 20 min |
| 9 · Low-rank approximation | Eckart–Young, denoising, ratings and drug responses | 15 min |
| 10 · Eigengenes of the infection experiment | SVD of the expression matrix = PCA; tissues, infection, genes | 15 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

The course's infection experiment measured **400 genes** in **24 tissue samples** — spleen, kidney and liver, each
from 4 control and 4 infected animals. That is a table of 9,600 numbers. A biologist looking at it asks:

* What are the **main patterns** in this table? Do the samples group by **tissue**, or by **infection**?
* **Which genes** drive each pattern — is there a coherent infection-response programme?
* How many **independent biological programmes** are hiding in the data, and how much is **noise**?
* If a measurement is **missing** (a failed assay, a drug not tested on a cell line), can we predict it from the rest?

Module 12 found the special directions of a *square* matrix. A data table is *rectangular* (24 × 400), and its rows
and columns live in different spaces — samples and genes. This module first turns eigenvectors into a factorisation
of square matrices (**diagonalization**, $A = X\Lambda X^{-1}$, which makes powers $A^k$ cheap — population projections
and Markov chains), and then generalises it to every matrix: the **singular value decomposition**
$A = U\Sigma V^\top$. In genomics the SVD was popularised by Alter, Brown and Botstein (2000), who called its singular
vectors **eigengenes** and **eigenarrays**; it is the computational heart of PCA, of noise filtering and of the
recommendation-style methods that fill in missing drug responses.

| Biological question | Mathematical tool | Where in this module |
|---|---|---|
| Project a population or a Markov chain $t$ steps ahead, in closed form | $A^t = X\Lambda^tX^{-1}$ | §5 |
| Which few expression patterns explain most of the data? | singular values $\sigma_1\ge\sigma_2\ge\dots$ | §6, §10 |
| Which samples are alike, which genes drive a pattern? | left and right singular vectors (eigengenes, eigenarrays) | §10 |
| How many independent programmes are there? | rank = number of non-zero singular values | §6, §9 |
| Remove measurement noise | truncated SVD (Eckart–Young) | §9 |
| Predict a missing rating or drug response | low-rank matrix completion | §9 |

> 🧪 **Simulated data, with a known truth.** The expression matrix is the course's simulated infection experiment: the
> 12 named infection-response genes (cytokines, interferon-stimulated genes, acute-phase proteins) respond in every
> tissue, other genes respond in one or two tissues, and a third of the genes differ in baseline level between tissues.
> Because we built it, we can check that the SVD finds "tissue" and "infection" as separate patterns and that the
> infection pattern is carried by the genes we planted. The denoising and drug-response examples are simulated from a
> known low-rank truth, so we can measure how close the SVD gets to it.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They simulate the infection experiment's expression matrix, draw 2-D arrows, print matrices neatly, and provide the background DNA Markov chain of Module 12. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_vectors(vectors: list, labels=None, colors=None, ax=None, lim=None, title: str | None = None,
                 origin=(0.0, 0.0)):
    """Draw 2-D vectors as arrows starting at ``origin`` (default: the origin of the plane)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.0, 4.0))
    default = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]
    colors = colors or [default[i % len(default)] for i in range(len(vectors))]
    ox, oy = origin
    for i, v in enumerate(vectors):
        ax.quiver(ox, oy, v[0], v[1], angles="xy", scale_units="xy", scale=1, color=colors[i], width=0.012)
        if labels:
            ax.text(ox + v[0] * 1.07, oy + v[1] * 1.07, labels[i], color=colors[i], fontsize=10,
                    ha="center", va="center")
    if lim is None:
        m = max([abs(c) for v in vectors for c in (ox + v[0], oy + v[1])] + [1.0]) * 1.25
        lim = (-m, m)
    ax.set_xlim(*lim)
    ax.set_ylim(*lim)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.axvline(0, color="0.6", lw=0.8)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    if title:
        ax.set_title(title)
    return ax


def format_matrix(M, n_coef=None, width: int = 7) -> str:
    """Pretty-print a matrix of numbers, Fractions or SymPy rationals (complex entries with a zero
    imaginary part print as reals); a bar ``|`` is drawn after ``n_coef`` columns."""
    M = np.asarray(M, dtype=object)

    def fmt(x):
        if getattr(x, "is_Rational", False):            # a SymPy Rational/Integer → exact fraction
            x = Fraction(int(x.p), int(x.q))
        if isinstance(x, Fraction):
            return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        if isinstance(x, (complex, np.complexfloating)):
            if abs(x.imag) > 1e-12:
                return f"{x.real:.3g}{x.imag:+.3g}i"
            x = x.real
        x = float(x)
        return "0" if abs(x) < 1e-12 else f"{x:.4g}"

    lines = []
    for row in M:
        cells = [fmt(x).rjust(width) for x in row]
        if n_coef is not None and n_coef < len(cells):
            cells = cells[:n_coef] + ["  |"] + cells[n_coef:]
        lines.append("[" + "".join(cells) + " ]")
    return "\n".join(lines)


BACKGROUND_TRANSITIONS = np.array([
    [0.30, 0.20, 0.29, 0.21],
    [0.32, 0.30, 0.08, 0.30],
    [0.25, 0.25, 0.29, 0.21],
    [0.18, 0.24, 0.29, 0.29],
])


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}

print("🧰 toolkit loaded:", ", ".join(sorted(['BACKGROUND_TRANSITIONS', 'HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'format_matrix', 'plot_vectors', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 The data of this module: the SIMULATED infection experiment, 24 samples × 400 genes ──
expr = simulate_tissue_expression()
X = expr["X"]                                       # N = 24 samples (rows) × J = 400 genes (columns), log2 scale
samples, genes = expr["samples"], np.array(expr["genes"])
print(expr["source"])
print("X:", X.shape, "→", X.size, "numbers")
print(samples.groupby(["tissue", "condition"]).size().unstack())

Xc = X - X.mean(axis=0)                             # centre every gene (column) at 0
fig, ax = plt.subplots(figsize=(10, 3.4))
im = ax.imshow(Xc, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
ax.set_yticks(range(0, 24, 4)); ax.set_yticklabels([f"{t} {c[:3]}" for t, c in
                                                     samples[["tissue", "condition"]].to_numpy()[::4]], fontsize=7)
ax.set_xlabel("gene (column) — the first 12 are the named infection-response genes")
ax.set_title("SIMULATED infection experiment: centred log2 expression (samples × genes)")
plt.colorbar(im, ax=ax, label="log2 expression − gene mean", fraction=0.03); plt.tight_layout(); plt.show()

🔍 **What just happened?** The matrix has 24 rows (samples: 4 control and 4 infected per tissue) and 400 columns
(genes), 9,600 numbers in all. After subtracting each gene's mean, the heat map shows some structure to the naked
eye: many columns change colour from one block of 8 rows to the next (the three tissues differ in the baseline level of
about a third of the genes), and in the first 12 columns the infected rows are red and the control rows blue (the
infection-response genes). But most of the
400 columns look like noise. Section 10 shows that three numbers per sample — three singular vectors — capture these
patterns, and tell us which genes carry them.

---
## 1 · Equivalent and similar matrices

> 📍 **Where we are.** In Module 11 you changed coordinates: the same vector has different coordinates in different
> bases, related by an invertible matrix. The same is true of a **linear map**: in a different basis it is represented
> by a different matrix. "Similar" matrices are two descriptions of the same map; this is the language in which
> diagonalization is stated.

**Definitions.**

* Two matrices $A, A'\in\mathbb R^{m\times n}$ are **equivalent** if there are nonsingular (invertible) matrices
  $S\in\mathbb R^{n\times n}$ and $T\in\mathbb R^{m\times m}$ such that

$$A' = T^{-1}AS .$$

  *In words:* $A'$ is $A$ seen with a new basis in the input space ($S$) and, independently, a new basis in the output
  space ($T$).

* Two **square** matrices $B, B'\in\mathbb R^{n\times n}$ are **similar** if there is one nonsingular
  $P\in\mathbb R^{n\times n}$ such that

$$B' = P^{-1}BP .$$

  *In words:* the same change of basis $P$ is used on the input and on the output side — $B'$ is the same linear map as
  $B$, written in the basis formed by the columns of $P$: convert coordinates from the $P$-basis to the standard basis
  ($P$), apply $B$, convert back ($P^{-1}$).

**Similar ⇒ equivalent, not conversely.** Choosing $S = T = P$ turns the equivalence formula into the similarity
formula, so every pair of similar matrices is equivalent. The converse fails: $I_2$ and $2I_2$ are equivalent
($T = I_2$, $S = 2I_2$ gives $T^{-1}I_2S = 2I_2$) but not similar, because $P^{-1}I_2P = P^{-1}P = I_2 \neq 2I_2$ for every
$P$. (In fact two matrices of the same size are equivalent exactly when they have the same rank — row and column
operations can bring any rank-$r$ matrix to the form $\left(\begin{smallmatrix}I_r & 0\\ 0 & 0\end{smallmatrix}\right)$.
Similarity preserves much more.)

**Theorem — similar matrices have the same characteristic polynomial**, hence the same eigenvalues (with
multiplicities), the same determinant and the same trace. Proof, one line at a time:

$$\begin{aligned}
\det(B' - \lambda I) &= \det(P^{-1}BP - \lambda P^{-1}P)\\
&= \det\big(P^{-1}(B - \lambda I)P\big)\\
&= \det(P^{-1})\,\det(B - \lambda I)\,\det(P) = \det(B - \lambda I).
\end{aligned}$$

*In words:* write $I = P^{-1}P$; factor $P^{-1}$ out on the left and $P$ on the right; the determinant of a product is
the product of determinants (Module 08), and $\det(P^{-1})\det(P) = \det(P^{-1}P) = 1$. Since the eigenvalues are the
roots of this polynomial, and $\det = \prod\lambda_i$, $\operatorname{tr} = \sum\lambda_i$ ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)), all of them agree.
The eigen*vectors* change: if $B\mathbf x = \lambda\mathbf x$ then $B'(P^{-1}\mathbf x) = P^{-1}BPP^{-1}\mathbf x = P^{-1}B\mathbf x
= \lambda P^{-1}\mathbf x$ — the eigenvector of $B'$ is $P^{-1}\mathbf x$, the same vector expressed in the new coordinates.

In [ ]:
# ── 1.1 A random change of basis leaves eigenvalues, determinant and trace unchanged ──
lam = sp.symbols("lambda")
B = sp.Matrix([[1, 1, -1], [2, 3, -4], [4, 1, -4]])             # the 3 × 3 example of Module 12
P = sp.Matrix([[1, 2, 0], [0, 1, 1], [1, 0, 1]])                  # any invertible matrix will do
print("det P =", P.det(), "(non-zero → invertible)")
B_sim = P.inv() * B * P
print("B' = P⁻¹ B P =", B_sim.tolist())
print("char. polynomial of B :", sp.factor(B.charpoly(lam).as_expr()))
print("char. polynomial of B':", sp.factor(B_sim.charpoly(lam).as_expr()))
print("det:", B.det(), "vs", B_sim.det(), "| trace:", B.trace(), "vs", B_sim.trace())

x = sp.Matrix([1, 7, 11])                                         # eigenvector of B for λ = −3
y = P.inv() * x                                                   # the same vector in the new coordinates
print(f"B x = {list(B * x)} = −3·x;   B'(P⁻¹x) = {list(B_sim * y)} = −3·{list(y)}")

I2 = sp.eye(2)
print("\nI₂ and 2I₂: ranks", I2.rank(), (2 * I2).rank(), "(equal → equivalent); eigenvalues",
      list(I2.eigenvals()), "vs", list((2 * I2).eigenvals()), "(different → not similar)")

🔍 **What just happened?** With an arbitrary invertible $P$ ($\det P = 3$), the matrix $B' = P^{-1}BP$ looks nothing
like $B$ — its entries are fractions — yet both have the characteristic polynomial
$(\lambda - 1)(\lambda - 2)(\lambda + 3)$, determinant $-6$ and trace 0. The eigenvector $(1, 7, 11)$ of $B$ (eigenvalue
$-3$) reappears as $P^{-1}(1,7,11)^\top = (3, -1, 8)$, an eigenvector of $B'$ with the same eigenvalue. And $I_2$, $2I_2$ have the same
rank (equivalent) but different eigenvalues (not similar).

The natural question: among all matrices similar to $A$, is there a particularly **simple** one — a diagonal one? That
is diagonalization.

✏️ **Your turn.** Show that similarity is *transitive*: if $B' = P^{-1}BP$ and $B'' = Q^{-1}B'Q$, then $B''$ is similar
to $B$. Which matrix plays the role of $P$? (Hint: $(PQ)^{-1} = Q^{-1}P^{-1}$, Module 09.)

### 📘 Lecture notes — equivalent and similar matrices

**Equivalent.** $A, A'\in\mathbb R^{m\times n}$ are equivalent if there exist nonsingular $S\in\mathbb R^{n\times n}$,
$T\in\mathbb R^{m\times m}$ with $A' = T^{-1}AS$ (two independent changes of basis; for matrices of the same size,
equivalent ⇔ same rank).

**Similar.** $B, B'\in\mathbb R^{n\times n}$ are similar if there exists a nonsingular $P\in\mathbb R^{n\times n}$ with
$B' = P^{-1}BP$ (one change of basis: the same linear map in another basis).

**Relation.** All similar matrix pairs are equivalent ($S = T = P$), but not the reverse ($I_2$ and $2I_2$).

**Invariants of similarity.** $\det(P^{-1}BP - \lambda I) = \det(P^{-1})\det(B-\lambda I)\det(P) = \det(B - \lambda I)$: same
characteristic polynomial, eigenvalues (with multiplicities), determinant, trace and rank. Eigenvectors transform as
$\mathbf x\mapsto P^{-1}\mathbf x$.

**Common confusion.** Similar matrices need not look alike entry by entry; equal eigenvalues do not by themselves make
two matrices similar ($\left(\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right)$ and $I_2$ both have eigenvalues 1, 1, but
$P^{-1}I_2P = I_2$ always).

> 🎓 **Exam tip.** To show two matrices are **not** similar, find one invariant that differs (trace, determinant,
> rank, eigenvalues). To show they **are** similar, exhibit $P$.

---
## 2 · Diagonalization

> 📍 **Where we are.** Diagonal matrices are the easiest matrices there are: their powers, inverse, determinant and
> eigenvalues can be read off. Diagonalization asks whether a matrix is a diagonal matrix *in disguise* — similar to
> one — and the answer is given entirely by its eigenvectors.

A **diagonal matrix** $D\in\mathbb R^{n\times n}$ has the value 0 in all off-diagonal positions:

$$D = \begin{pmatrix}c_1 & \cdots & 0\\ \vdots & \ddots & \vdots\\ 0 & \cdots & c_n\end{pmatrix} = \operatorname{diag}(c_1,\dots,c_n),
\qquad D^k = \operatorname{diag}(c_1^k,\dots,c_n^k).$$

*In words:* a diagonal matrix just multiplies coordinate $i$ by $c_i$; applying it $k$ times multiplies by $c_i^k$.

**Definition.** $A\in\mathbb R^{n\times n}$ is **diagonalizable** if it is similar to a diagonal matrix, i.e. there is a
nonsingular $P\in\mathbb R^{n\times n}$ such that $D = P^{-1}AP$ is diagonal.

**Key lemma.** Let $\lambda_1,\dots,\lambda_n$ be scalars, $\mathbf p_1,\dots,\mathbf p_n$ vectors in $\mathbb R^n$,
$P = [\mathbf p_1,\dots,\mathbf p_n]$ (the $\mathbf p_i$ are the **columns**) and $D = \operatorname{diag}(\lambda_1,\dots,\lambda_n)$. Then

$$AP = PD \iff A\mathbf p_i = \lambda_i\mathbf p_i \text{ for every } i = 1,\dots,n .$$

*In words:* the single matrix equation $AP = PD$ says exactly that every column $\mathbf p_i$ is mapped by $A$ to
$\lambda_i$ times itself.

**Proof, column by column.** Column $i$ of a product $MN$ is $M$ times column $i$ of $N$. Column $i$ of $AP$ is therefore
$A\mathbf p_i$. Column $i$ of $PD$ is $P$ times column $i$ of $D$, which is $\lambda_i\mathbf e_i$ (only entry $i$ is
non-zero), so it equals $\lambda_iP\mathbf e_i = \lambda_i\mathbf p_i$ — multiplying by a diagonal matrix on the right
scales the columns. Two matrices are equal iff all their columns are equal:

$$AP = [A\mathbf p_1,\dots,A\mathbf p_n] \overset{?}{=} [\lambda_1\mathbf p_1,\dots,\lambda_n\mathbf p_n] = PD .$$

*In words:* the single matrix equation $AP = PD$ is just the $n$ eigenvalue equations written side by side — so
(with non-zero columns) the $\lambda_i$ are eigenvalues of $A$ and the $\mathbf p_i$ the corresponding eigenvectors. ∎

**Eigendecomposition.** Collect the eigenvectors of $A$ as the columns of $X$ and the eigenvalues on the diagonal of
$\Lambda = \operatorname{diag}(\lambda_1,\dots,\lambda_n)$, in the same order. Then $AX = X\Lambda$ (it satisfies all the
eigenvalue equations simultaneously). **If the eigenvectors are linearly independent**, $X$ is invertible (Module 11),
and multiplying by $X^{-1}$ on the right gives

$$A = X\Lambda X^{-1}, \qquad \Lambda = X^{-1}AX .$$

*In words:* $A$ is similar to the diagonal matrix of its eigenvalues, with the eigenvector matrix as the change of
basis. This factorisation is the **eigendecomposition** of $A$, and it exists exactly when $A$ is diagonalizable:

$$A \text{ is diagonalizable} \iff A \text{ has } n \text{ linearly independent eigenvectors (a basis of } \mathbb R^n\text{)}.$$

*In words:* a diagonal "disguise" exists exactly when the eigenvectors are numerous enough to serve as a basis — they
are then the new coordinate axes in which $A$ only stretches.

⚠️ **Common confusion — eigenvectors, not eigenvalues, must form a basis.** Eigenvalues are numbers; it is the
eigen**vectors** that must be $n$ independent vectors. A sufficient condition is $n$ distinct eigenvalues ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)); with
repeated eigenvalues you must count independent eigenvectors (geometric multiplicities).

In [ ]:
# ── 2.1 AP = PD column by column, for A = [[3,4],[2,1]] (eigenvalues 5, −1 from Module 12) ──
A = sp.Matrix([[3, 4], [2, 1]])
Pm = sp.Matrix([[2, 1], [1, -1]])                  # columns: eigenvectors (2,1) for λ = 5 and (1,−1) for λ = −1
Dm = sp.diag(5, -1)
for i in range(2):
    print(f"column {i + 1}:  A·p{i + 1} = {list(A * Pm[:, i])}   λ{i + 1}·p{i + 1} = {list(Dm[i, i] * Pm[:, i])}")
print("AP =", (A * Pm).tolist(), "  PD =", (Pm * Dm).tolist())
print("X Λ X⁻¹ =", (Pm * Dm * Pm.inv()).tolist(), "= A ?", Pm * Dm * Pm.inv() == A)
print("X⁻¹ A X =", (Pm.inv() * A * Pm).tolist(), "(the diagonal matrix Λ)")

wrong_order = sp.diag(-1, 5)                       # eigenvalues listed in the wrong order
print("with Λ in the wrong order: X Λ X⁻¹ =", (Pm * wrong_order * Pm.inv()).tolist())

🔍 **What just happened?** Column by column, $A\mathbf p_1 = (10, 5) = 5\mathbf p_1$ and $A\mathbf p_2 = (-1, 1) = -1\cdot\mathbf p_2$,
so $AP = PD = \begin{pmatrix}10&-1\\5&1\end{pmatrix}$; since $\det P = -3 \neq 0$, $P$ is invertible and
$P D P^{-1} = \begin{pmatrix}3&4\\2&1\end{pmatrix} = A$ exactly, while $P^{-1}AP = \operatorname{diag}(5,-1)$. The last line
shows the bookkeeping rule: the $i$-th eigenvalue in $\Lambda$ must belong to the $i$-th column of $X$. Swapping the
eigenvalues but not the columns produces a different matrix, $\begin{pmatrix}1&-4\\-2&3\end{pmatrix}$.

### 📘 Lecture notes — diagonalization and the eigendecomposition

**Diagonal matrix.** $D\in\mathbb R^{n\times n}$ with 0 in all off-diagonal locations, $D = \operatorname{diag}(c_1,\dots,c_n)$.

**Diagonalizable.** $A\in\mathbb R^{n\times n}$ is diagonalizable if it is similar to a diagonal matrix: there exists a
nonsingular $P$ with $D = P^{-1}AP$.

**Lemma.** With $P = [\mathbf p_1,\dots,\mathbf p_n]$ (columns) and $D = \operatorname{diag}(\lambda_1,\dots,\lambda_n)$:
$AP = PD$ iff $\lambda_1,\dots,\lambda_n$ are eigenvalues of $A$ and $\mathbf p_1,\dots,\mathbf p_n$ corresponding eigenvectors
(column $i$: $A\mathbf p_i = \lambda_i\mathbf p_i$).

**Eigendecomposition.** $X$ = eigenvectors as columns, $\Lambda = \langle\lambda_1,\dots,\lambda_n\rangle I_n$ = eigenvalues on
the diagonal: $AX = X\Lambda$ always; if the eigenvectors are linearly independent, $X$ is invertible and

$$A = X\Lambda X^{-1} .$$

*In words:* $A$ can be factored into eigenvector matrix × diagonal eigenvalue matrix × inverse eigenvector matrix
exactly when its eigen**vectors** form a basis of $\mathbb R^n$ — this is what "diagonalizable" means.

**Sufficient condition.** $n$ distinct eigenvalues ⇒ diagonalizable. **Obstruction.** An eigenvalue whose geometric
multiplicity (number of independent eigenvectors) is smaller than its algebraic multiplicity.

**Common confusions.** The order of the eigenvalues in $\Lambda$ must match the order of the columns of $X$. Any non-zero
scaling of the columns of $X$ gives another valid eigendecomposition. "The eigenvalues form a basis" is meaningless —
eigenvectors form the basis.

> 🎓 **Exam tip.** To diagonalize: (1) eigenvalues from $\det(A-\lambda I)=0$; (2) eigenvectors; (3) check you have $n$
> independent ones; (4) write $X$, $\Lambda$, compute $X^{-1}$; (5) verify $AX = X\Lambda$ (no inverse needed for the check).

---
## 3 · Worked eigendecompositions — and matrices that refuse

> 📍 **Where we are.** The recipe is: eigenvalues, eigenvectors, assemble $X$ and $\Lambda$, invert $X$. We do it for a
> symmetric 2 × 2 matrix, a non-symmetric one, the 3 × 3 matrix of [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) — and then meet matrices for which step 3
> fails.

### 3.1 Perform the eigendecomposition of $A = \begin{pmatrix}2&1\\1&2\end{pmatrix}$

**Eigenvalues.** $\det(A - \lambda I) = (2-\lambda)^2 - 1 = \lambda^2 - 4\lambda + 3 = (\lambda - 3)(\lambda - 1)$, so
$\lambda_1 = 3$, $\lambda_2 = 1$.

**Eigenvectors.** $\lambda_1 = 3$: $\begin{pmatrix}-1&1\\1&-1\end{pmatrix}\mathbf x = \mathbf 0 \Rightarrow x_1 = x_2$,
$\mathbf x_1 = (1,1)$. $\lambda_2 = 1$: $\begin{pmatrix}1&1\\1&1\end{pmatrix}\mathbf x = \mathbf 0\Rightarrow x_1 = -x_2$,
$\mathbf x_2 = (1,-1)$.

**Assemble.** $X = \begin{pmatrix}1&1\\1&-1\end{pmatrix}$, $\det X = -1 - 1 = -2$, and by the 2 × 2 inverse formula
(Module 09) $X^{-1} = \frac{1}{-2}\begin{pmatrix}-1&-1\\-1&1\end{pmatrix} = \frac12\begin{pmatrix}1&1\\1&-1\end{pmatrix}$:

$$A = X\Lambda X^{-1} = \begin{pmatrix}1&1\\1&-1\end{pmatrix}\begin{pmatrix}3&0\\0&1\end{pmatrix}\frac12\begin{pmatrix}1&1\\1&-1\end{pmatrix}
= \frac12\begin{pmatrix}3&1\\3&-1\end{pmatrix}\begin{pmatrix}1&1\\1&-1\end{pmatrix} = \frac12\begin{pmatrix}4&2\\2&4\end{pmatrix}
= \begin{pmatrix}2&1\\1&2\end{pmatrix}.$$

*In words:* first $X\Lambda$ scales the columns of $X$ by 3 and 1, then multiplying by $X^{-1}$ gives back $A$.

**The symmetric shortcut.** Because $A$ is symmetric its eigenvectors are orthogonal; normalising them to length 1
gives an **orthogonal** $P = \frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ with $P^{-1} = P^\top$, so
$A = PDP^\top$ — the spectral theorem of [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb). No inverse has to be computed.

In [ ]:
# ── 3.1 The same eigendecomposition with SymPy (exact) and NumPy (floating point) ──
A_s = sp.Matrix([[2, 1], [1, 2]])
X_s = sp.Matrix([[1, 1], [1, -1]])
L_s = sp.diag(3, 1)
print("X⁻¹ =", X_s.inv().tolist())
print("X Λ X⁻¹ =", (X_s * L_s * X_s.inv()).tolist())
P_orth = X_s / sp.sqrt(2)
print("P = X/√2 orthogonal?  PᵀP =", (P_orth.T * P_orth).tolist(), ";  P D Pᵀ =", (P_orth * L_s * P_orth.T).tolist())

P_sympy, D_sympy = A_s.diagonalize()               # SymPy's own choice of eigenvectors and order
print("SymPy diagonalize(): P =", P_sympy.tolist(), " D =", D_sympy.tolist())

vals, vecs = np.linalg.eig(np.array([[2.0, 1.0], [1.0, 2.0]]))
print("NumPy eig: λ =", vals, "\nX =\n", vecs.round(4), "\nX Λ X⁻¹ =\n", (vecs @ np.diag(vals) @ np.linalg.inv(vecs)).round(12))

🔍 **What just happened?** SymPy confirms $X^{-1} = \tfrac12\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ and $X\Lambda X^{-1} = A$
exactly; with the normalised $P = X/\sqrt2$, $P^\top P = I$ and $PDP^\top = A$. SymPy's `diagonalize()` makes other
(equally valid) choices — eigenvalue 1 first, eigenvector $(-1, 1)$ — and NumPy's `eig` returns unit-length columns
$\pm(0.7071, 0.7071)$ and $(-0.7071, 0.7071)$. An eigendecomposition is unique only up to the order of the eigenvalues
and the scaling of each eigenvector; every valid choice reproduces $A$.

### 3.2 A non-symmetric example, and what "powers" look like in eigen-coordinates

$A = \begin{pmatrix}1&2\\3&2\end{pmatrix}$ has the eigenelements $\mathbf v_1 = (1, \tfrac32)$ with $\lambda_1 = 4$ and
$\mathbf v_2 = (4,-4)$ with $\lambda_2 = -1$ (check: $A\mathbf v_1 = (1 + 3,\ 3 + 3) = (4, 6) = 4\mathbf v_1$ and
$A\mathbf v_2 = (4 - 8,\ 12 - 8) = (-4, 4) = -\mathbf v_2$). In [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) we watched what $A$ does to $\mathbf u = (1,2)$:
$A\mathbf u = (5,7)$ and $A(A\mathbf u) = (19, 29)$. Diagonalization explains these numbers. First write $\mathbf u$ in the
eigenbasis by solving $X\mathbf c = \mathbf u$:

$$\mathbf u = \begin{pmatrix}1\\2\end{pmatrix} = c_1\begin{pmatrix}1\\ 3/2\end{pmatrix} + c_2\begin{pmatrix}4\\-4\end{pmatrix}
\ \Rightarrow\ \begin{cases}c_1 + 4c_2 = 1\\ \tfrac32c_1 - 4c_2 = 2\end{cases}\ \Rightarrow\ c_1 = 1.2,\ c_2 = -0.05 .$$

*In words:* add the two equations to eliminate $c_2$: $\tfrac52c_1 = 3$, so $c_1 = 1.2$, and then $c_2 = (1 - 1.2)/4 = -0.05$.

Then each application of $A$ simply multiplies each eigen-coordinate by its eigenvalue:

$$A^k\mathbf u = 1.2\cdot4^k\,\mathbf v_1 - 0.05\cdot(-1)^k\,\mathbf v_2 ,$$

*In words:* in eigen-coordinates the matrix acts like the diagonal matrix $\operatorname{diag}(4, -1)$. For $k = 1$:
$4.8\,(1, 1.5) + 0.05\,(4, -4) = (4.8 + 0.2,\ 7.2 - 0.2) = (5, 7)$; for $k = 2$: $19.2\,(1,1.5) - 0.05\,(4,-4) =
(19.2 - 0.2,\ 28.8 + 0.2) = (19, 29)$. The $\mathbf v_1$ part grows like $4^k$ and swamps the $\mathbf v_2$ part — which is
why power iteration converged to $\mathbf v_1$.

In [ ]:
# ── 3.2 Eigen-coordinates of u = (1, 2) and the powers A^k u ──
A2 = np.array([[1.0, 2.0], [3.0, 2.0]])
X2 = np.array([[1.0, 4.0], [1.5, -4.0]])            # columns v1 = (1, 3/2) and v2 = (4, −4)
lams2 = np.array([4.0, -1.0])
u = np.array([1.0, 2.0])
c = np.linalg.solve(X2, u)                          # eigen-coordinates: X c = u
print("eigen-coordinates c =", c)
for k in range(1, 5):
    via_eigen = X2 @ (lams2 ** k * c)               # Σ c_i λ_i^k v_i
    direct = np.linalg.matrix_power(A2, k) @ u
    print(f"k = {k}:  Σ c_i λ_i^k v_i = {via_eigen}   A^k u = {direct}")
print("A = X Λ X⁻¹ ?", np.allclose(X2 @ np.diag(lams2) @ np.linalg.inv(X2), A2))

🔍 **What just happened?** The eigen-coordinates of $\mathbf u$ are $(1.2, -0.05)$, and $\sum_ic_i\lambda_i^k\mathbf v_i$
reproduces $A^k\mathbf u$ exactly: $(5,7)$, $(19,29)$, $(77,115)$, $(307,461)$ — the iterates of [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)'s power
iteration. Computing $A^k\mathbf u$ by repeated multiplication and by "convert to eigen-coordinates, raise the eigenvalues to
the $k$-th power, convert back" are the same thing.

### 3.3 A 3 × 3 eigendecomposition

[Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) found the eigenpairs of $B = \begin{pmatrix}1&1&-1\\2&3&-4\\4&1&-4\end{pmatrix}$: $(1;\ (1,1,1))$, $(2;\ (1,2,1))$ and
$(-3;\ (1,7,11))$. Three distinct eigenvalues guarantee three independent eigenvectors, so $B$ is diagonalizable with

$$X = \begin{pmatrix}1&1&1\\1&2&7\\1&1&11\end{pmatrix},\qquad \Lambda = \begin{pmatrix}1&0&0\\0&2&0\\0&0&-3\end{pmatrix}.$$

*In words:* the eigenvectors are the columns of $X$, in the same order as the eigenvalues on the diagonal of $\Lambda$.

In [ ]:
# ── 3.3 B = X Λ X⁻¹ exactly ──
B = sp.Matrix([[1, 1, -1], [2, 3, -4], [4, 1, -4]])
X3 = sp.Matrix([[1, 1, 1], [1, 2, 7], [1, 1, 11]])
L3 = sp.diag(1, 2, -3)
print("det X =", X3.det())
print("X⁻¹ =", X3.inv().tolist())
print("B X == X Λ ?", B * X3 == X3 * L3, "   X Λ X⁻¹ == B ?", X3 * L3 * X3.inv() == B)

🔍 **What just happened?** $\det X = 10 \neq 0$, so the three eigenvectors are independent and $X$ is invertible; its
inverse has fractions with denominator 10, and $X\Lambda X^{-1}$ reproduces $B$ exactly. (Checking $BX = X\Lambda$ first is
cheaper — no inverse needed.)

### 3.4 Matrices that cannot be diagonalized

* **The shear** $S = \begin{pmatrix}1&1\\0&1\end{pmatrix}$: triangular, so $\lambda = 1$ twice; but
  $S - I = \begin{pmatrix}0&1\\0&0\end{pmatrix}$ forces $x_2 = 0$, so every eigenvector is a multiple of $(1,0)$. One
  independent eigenvector for a 2 × 2 matrix: no invertible $X$, no eigendecomposition.
* **The insect population matrix** $L = \begin{pmatrix}0&6&8\\\frac12&0&0\\0&\frac12&0\end{pmatrix}$ of [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb): eigenvalues
  2 and $-1$ (double), but the double eigenvalue has only the eigenvector $(4,-2,1)$ — 2 independent eigenvectors in
  $\mathbb R^3$.
* **A rotation** by 30° has complex eigenvalues $0.87\pm0.5i$: it is not diagonalizable with *real* matrices, but it is
  over the complex numbers (complex $X$ and $\Lambda$).

Numerical libraries do not refuse: `np.linalg.eig` still returns an "eigenvector matrix", but its columns are (almost)
parallel and the matrix is (almost) singular — its **condition number** $\|X\|\,\|X^{-1}\|$ is huge.

In [ ]:
# ── 3.4 Diagonalizable or not? SymPy decides exactly; NumPy shows an ill-conditioned X ──
L_insect = sp.Matrix([[0, 6, 8], [sp.Rational(1, 2), 0, 0], [0, sp.Rational(1, 2), 0]])
cases = {"shear [[1,1],[0,1]]": sp.Matrix([[1, 1], [0, 1]]),
         "insect Leslie L": L_insect,
         "B (3 distinct λ)": B,
         "rotation 90° [[0,-1],[1,0]]": sp.Matrix([[0, -1], [1, 0]])}
for name, M in cases.items():
    geo = {str(val): len(vs) for val, _, vs in M.eigenvects()}
    alg = {str(k): v for k, v in M.eigenvals().items()}
    real_ok = M.is_diagonalizable(reals_only=True)
    _, vecs = np.linalg.eig(np.array(M, dtype=float))
    print(f"{name:28s} algebraic {alg}  geometric {geo}  diagonalizable over ℝ: {real_ok!s:5s}  "
          f"cond(X) from NumPy = {np.linalg.cond(vecs):.1e}")

🔍 **What just happened?** For the shear, the eigenvalue 1 has algebraic multiplicity 2 but geometric multiplicity 1,
SymPy answers "not diagonalizable", and NumPy's eigenvector matrix has a condition number around $10^{15}$–$10^{16}$ —
numerically singular (its two columns are parallel up to rounding). The insect Leslie matrix fails in the same way
for $\lambda = -1$ (algebraic 2, geometric 1); NumPy's $X$ is badly conditioned too. $B$, with three distinct
eigenvalues, is diagonalizable with a well-conditioned $X$. The rotation by 90° has eigenvalues $\pm i$: no real
diagonalization (`reals_only=True` says False), although its complex eigenvector matrix is perfectly conditioned.
Non-diagonalizable matrices still have a "nearly diagonal" form, the Jordan normal form (see the 🚀 Bonus); for this
course, remember the test: *count independent eigenvectors*.

✏️ **Your turn.** Diagonalize $\begin{pmatrix}4&1\\2&3\end{pmatrix}$ by hand (eigenvalues 5 and 2, from [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)'s checkpoint) and
check with `sp.Matrix(...).diagonalize()`. Then decide, without computing eigenvectors, whether
$\begin{pmatrix}2&0&0\\0&3&4\\0&0&5\end{pmatrix}$ is diagonalizable.

---
## 4 · The geometry: change to eigen-coordinates, stretch, change back

> 📍 **Where we are.** $A = PDP^{-1}$ is three simple maps in a row. Reading the product from **right to left** (the
> order in which they act on a vector $\mathbf x$):

$$A\mathbf x = P\,\big(D\,(P^{-1}\mathbf x)\big).$$

*In words:* (1) $P^{-1}$ converts $\mathbf x$ from standard coordinates into **eigen-coordinates** (Module 11: the
columns of $P$ form the new basis); (2) $D$ **stretches** each eigen-coordinate by its eigenvalue — a plain
axis-by-axis scaling; (3) $P$ converts back to standard coordinates.

When $A$ is **symmetric**, $P$ can be chosen **orthogonal** ($P^{-1} = P^\top$): an orthogonal matrix with determinant
$+1$ is a **rotation**, so $A = PDP^\top$ reads "rotate the eigenvectors onto the coordinate axes, stretch along the
axes, rotate back". The next figure follows the unit circle and the basis vectors $\mathbf e_1, \mathbf e_2$ through the
three steps for $A = \begin{pmatrix}2&1\\1&2\end{pmatrix}$, with $\mathbf p_1 = (1,1)/\sqrt2$ ($\lambda_1 = 3$) and
$\mathbf p_2 = (-1,1)/\sqrt2$ ($\lambda_2 = 1$).

In [ ]:
# ── 4.1 A = P D Pᵀ step by step: unit circle → eigen-coordinates → stretch → back (A = [[2,1],[1,2]]) ──
A_g = np.array([[2.0, 1.0], [1.0, 2.0]])
p1, p2 = np.array([1.0, 1.0]) / np.sqrt(2), np.array([-1.0, 1.0]) / np.sqrt(2)
P_g = np.column_stack([p1, p2])                     # det = +1 → a rotation by 45°
D_g = np.diag([3.0, 1.0])
print("det P =", round(np.linalg.det(P_g), 6), "| P D Pᵀ =\n", (P_g @ D_g @ P_g.T).round(12))

phi = np.linspace(0, 2 * np.pi, 120)
circle = np.vstack([np.cos(phi), np.sin(phi)])
col = plt.cm.hsv(phi / (2 * np.pi))                 # colour = position on the circle, to follow each point
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
stages = [("x (standard coordinates)", np.eye(2)),
          ("Pᵀx = P⁻¹x: eigen-coordinates", P_g.T),
          ("D Pᵀx: stretch ×3 and ×1", D_g @ P_g.T),
          ("P D Pᵀx = Ax: back", A_g)]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, (title, M) in zip(axes, stages):
    pts = M @ circle
    ax.scatter(pts[0], pts[1], c=col, s=6)
    plot_vectors([M @ e1, M @ e2], colors=["tab:blue", "tab:blue"], ax=ax, lim=(-3.3, 3.3))
    plot_vectors([M @ p1, M @ p2], colors=["tab:orange", "tab:orange"], ax=ax, lim=(-3.3, 3.3))
    ax.set_title(title, fontsize=9)
axes[0].text(0.75, 0.75, "p1", color="tab:orange"); axes[0].text(-1.1, 0.75, "p2", color="tab:orange")
axes[0].text(1.05, -0.35, "e1", color="tab:blue"); axes[0].text(0.1, 1.1, "e2", color="tab:blue")
axes[2].text(2.4, 0.3, "λ1 = 3", color="tab:orange"); axes[2].text(0.1, 1.15, "λ2 = 1", color="tab:orange")
axes[3].text(1.6, 2.45, "Ap1 = 3p1", color="tab:orange", fontsize=8); axes[3].text(2.1, 0.6, "Ae1", color="tab:blue")
fig.suptitle("blue arrows: images of e1, e2;  orange arrows: images of the eigenvectors p1, p2;  dots: unit circle",
             fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** $P$ has determinant $+1$: it is the rotation by 45°, and $PDP^\top$ reproduces $A$. Follow
the orange arrows: in panel 1 they are the eigenvectors $\mathbf p_1$, $\mathbf p_2$ on the unit circle; $P^\top$ rotates
them onto the coordinate axes (panel 2 — now $\mathbf p_1$ sits on $\mathbf e_1$ and $\mathbf p_2$ on $\mathbf e_2$, and the
blue standard basis vectors have turned by $-45°$); $D$ stretches the horizontal axis by 3 and leaves the vertical one
alone, turning the circle into an axis-aligned ellipse (panel 3); $P$ rotates everything back by $+45°$ (panel 4). The
result is $A\mathbf x$: an ellipse whose long axis is $\mathbf p_1$ (stretched to $3\mathbf p_1$), while
$A\mathbf e_1 = (2,1)$ points elsewhere. The colours show that every point of the circle travels along consistently.

For a **non-symmetric** diagonalizable matrix, e.g. $\begin{pmatrix}1&2\\3&2\end{pmatrix}$, the same three steps hold, but
$P$ is not a rotation: its columns $(1, \tfrac32)$ and $(4, -4)$ are not perpendicular, so step 1 *shears* the plane
into skewed eigen-coordinates. And a matrix that is not diagonalizable has no such description at all — that gap is
filled by the SVD (Section 6), which uses **two different** rotations and works for every matrix.

### 📘 Lecture notes — the geometry of $A = PDP^{-1}$ and matrix powers

**Three steps.** $A\mathbf x = P(D(P^{-1}\mathbf x))$: convert to eigen-coordinates ($P^{-1}$), scale coordinate $i$ by
$\lambda_i$ ($D$), convert back ($P$). Symmetric $A$: $P$ orthogonal (a rotation), $A = PDP^\top$ = rotate, stretch,
rotate back; the unit circle becomes an ellipse with semi-axes $|\lambda_i|$ along $\mathbf p_i$.

**Powers.** $A^k = (PDP^{-1})(PDP^{-1})\cdots(PDP^{-1}) = PD^kP^{-1}$, because every inner $P^{-1}P$ cancels;
$D^k = \operatorname{diag}(\lambda_1^k,\dots,\lambda_n^k)$. In eigen-coordinates $\mathbf c = P^{-1}\mathbf x$:
$A^k\mathbf x = \sum_ic_i\lambda_i^k\mathbf p_i$.

*In words:* $k$ applications of $A$ become $k$-th powers of numbers; for large $k$ the term with the largest $|\lambda_i|$
dominates (if $c_i\neq0$).

**Worked example.** $A = \begin{pmatrix}1&2\\3&2\end{pmatrix}$, $\mathbf v_1 = (1,\frac32)$, $\lambda_1 = 4$;
$\mathbf v_2 = (4,-4)$, $\lambda_2 = -1$; $\mathbf u = (1,2) = 1.2\mathbf v_1 - 0.05\mathbf v_2$; $A\mathbf u = (5,7)$,
$A(A\mathbf u) = (19,29)$, $A^k\mathbf u = 1.2\cdot4^k\mathbf v_1 - 0.05(-1)^k\mathbf v_2$.

**Common confusion.** $A^k \neq P^kD^kP^{-k}$ — only the middle factor is powered. $P^{-1}$ acts *first* on $\mathbf x$.

> 🎓 **Exam tip.** For "compute $A^{10}$" or "what happens to $A^k\mathbf x$ as $k\to\infty$", diagonalize first: the
> answer is $PD^{10}P^{-1}$, and the long-run behaviour is $c_1\lambda_1^k\mathbf p_1$ for the dominant eigenvalue.

---
## 5 · Matrix powers: populations and Markov chains in closed form

> 📍 **Where we are.** [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) projected an insect population year by year, $\mathbf n_t = L^t\mathbf n_0$, and found the
> long-run growth factor and age structure by power iteration. Diagonalization gives the **exact formula** for every
> year at once.

The insect matrix of [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) is not diagonalizable (Section 3.4). Consider instead the same species in a poorer habitat,
where one-year-olds produce 3.5 and two-year-olds 3 surviving offspring (survival still ½ and ½; illustrative numbers):

$$L = \begin{pmatrix}0 & 3.5 & 3\\ \tfrac12 & 0 & 0\\ 0 & \tfrac12 & 0\end{pmatrix},\qquad
\det(L - \lambda I) = -\left(\lambda^3 - \tfrac74\lambda - \tfrac34\right) = -(\lambda - \tfrac32)(\lambda + \tfrac12)(\lambda + 1).$$

*In words:* expanding along column 3 as in [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) gives $3\cdot\tfrac14 - \lambda(\lambda^2 - \tfrac74)$; the roots
$\tfrac32, -\tfrac12, -1$ are distinct, so $L$ is diagonalizable.

For a Leslie matrix with survival $S_0, S_1$, an eigenvector for $\lambda$ is $(1,\ S_0/\lambda,\ S_0S_1/\lambda^2)$ (each
class is the survivors of the class before, one year earlier, divided by the growth factor). Scaled to integers:
$\lambda = \tfrac32 \to (9,3,1)$, $\lambda = -\tfrac12 \to (1,-1,1)$, $\lambda = -1 \to (4,-2,1)$. Starting from ten
two-year-olds, $\mathbf n_0 = (0,0,10)$, the eigen-coordinates $\mathbf c = X^{-1}\mathbf n_0$ give the closed form

$$\mathbf n_t = c_1\left(\tfrac32\right)^t\begin{pmatrix}9\\3\\1\end{pmatrix} + c_2\left(-\tfrac12\right)^t\begin{pmatrix}1\\-1\\1\end{pmatrix}
+ c_3(-1)^t\begin{pmatrix}4\\-2\\1\end{pmatrix}.$$

*In words:* the population is a sum of three "modes": one grows by 50 % per year with the stable age structure
$(9,3,1)/13$, one halves and flips sign every year, one flips sign every year without shrinking — but, relative to the
growing mode, both others fade.

In [ ]:
# ── 5.1 Closed-form projection n_t = X Λ^t X⁻¹ n0 vs year-by-year multiplication (illustrative population) ──
L_poor = sp.Matrix([[0, sp.Rational(7, 2), 3], [sp.Rational(1, 2), 0, 0], [0, sp.Rational(1, 2), 0]])
print("det(L − λI) =", sp.factor((L_poor - lam * sp.eye(3)).det()))
X_L = sp.Matrix([[9, 1, 4], [3, -1, -2], [1, 1, 1]])
Lam_L = sp.diag(sp.Rational(3, 2), -sp.Rational(1, 2), -1)
print("L X == X Λ ?", L_poor * X_L == X_L * Lam_L)
n0 = sp.Matrix([0, 0, 10])
c_L = X_L.inv() * n0
print("eigen-coordinates c = X⁻¹ n0 =", list(c_L))

t = sp.symbols("t", integer=True, nonnegative=True)
n_t = X_L * sp.diag(sp.Rational(3, 2) ** t, (-sp.Rational(1, 2)) ** t, (-1) ** t) * c_L
print("closed form n_t =", [sp.simplify(e) for e in n_t])

Lf, n = np.array(L_poor, dtype=float), np.array([0.0, 0.0, 10.0])
rows = []
for year in range(0, 13):
    closed = np.array([float(e.subs(t, year)) for e in n_t])
    rows.append(dict(year=year, iterated=n.round(3).tolist(), closed_form=closed.round(3).tolist(),
                     total=n.sum(), growing_mode_over_total=float(c_L[0]) * 1.5**year * 13 / n.sum()))
    n = Lf @ n
print(pd.DataFrame(rows).set_index("year").round(3).to_string())

🔍 **What just happened?** SymPy confirms the characteristic polynomial $-(\lambda - \tfrac32)(\lambda + \tfrac12)(\lambda+1)$
(printed with integer coefficients as $-(2\lambda - 3)(2\lambda + 1)(\lambda + 1)/4$) and $LX = X\Lambda$. The
eigen-coordinates of $\mathbf n_0 = (0,0,10)$ are $\mathbf c = (1, 15, -6)$ — check the first entry by hand:
$1\cdot(9,3,1) + 15\cdot(1,-1,1) - 6\cdot(4,-2,1) = (9 + 15 - 24,\ 3 - 15 + 12,\ 1 + 15 - 6) = (0, 0, 10)$ ✓. So

$$\mathbf n_t = \left(\tfrac32\right)^t(9,3,1) + 15\left(-\tfrac12\right)^t(1,-1,1) - 6(-1)^t(4,-2,1),$$

*In words:* one growing mode with weight 1, one shrinking-and-flipping mode with weight 15, and one flipping mode with
weight $-6$.

SymPy prints the same formula over the common denominator $2^t$ (e.g. the first entry
$9\cdot1.5^t + 15(-0.5)^t - 24(-1)^t = 3\,(3\cdot3^t + 5(-1)^t - 8(-2)^t)/2^t$), and the table shows the closed form agreeing
with year-by-year multiplication in every year. The last column divides the size of the growing mode,
$13\cdot1.5^t$ animals, by the total: it starts at 1.3 and swings (0.65, 1.95, 0.73, 1.35, …) because the other two
modes add or subtract animals with alternating signs, then settles towards 1 (0.984, 1.011 in years 11–12): the
population *becomes* the growing mode. With the closed form you can jump to year 100 without computing years 1–99.

### 🌉 Bridge — population projection and Markov chains

The closed form $\mathbf n_t = \sum_ic_i\lambda_i^t\mathbf x_i$ is how demographers and conservation biologists analyse
matrix population models (Caswell 2001): the dominant eigenvalue is the asymptotic growth rate, the ratio
$|\lambda_2|/\lambda_1$ (the *damping ratio*) says how fast transients die out, and the left and right eigenvectors give
reproductive values and stable stage structures. The same algebra governs **Markov chains** ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb)): a transition matrix
$P$ has the eigenvalue 1, all others satisfy $|\lambda|\le1$, and when they are strictly smaller than 1 in size (as for
any chain whose transition probabilities are all positive) $P^n = X\Lambda^nX^{-1}$ converges to a matrix whose rows all
equal the stationary distribution — the base composition [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) found as an eigenvector. The cell below shows
it for the background DNA chain, and the 02-620 companion uses the same machinery for hidden Markov models —
[BiologyToML · Module 19 — HMM inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb).

In [ ]:
# ── 5.2 Pⁿ of the background DNA Markov chain by diagonalization: every row → the stationary distribution ──
P_dna = BACKGROUND_TRANSITIONS
vals_P, X_P = np.linalg.eig(P_dna)                  # complex eigenvalues appear in conjugate pairs
X_P_inv = np.linalg.inv(X_P)
print("eigenvalues of P:", np.round(vals_P, 3), "  |λ| =", np.abs(vals_P).round(3))
for n_steps in [1, 2, 5, 20]:
    Pn = (X_P @ np.diag(vals_P ** n_steps) @ X_P_inv).real
    same = np.allclose(Pn, np.linalg.matrix_power(P_dna, n_steps))
    print(f"\nP^{n_steps} via X Λ^n X⁻¹ (= matrix_power? {same}):\n{format_matrix(Pn.round(4))}")

🔍 **What just happened?** The background chain has eigenvalue 1, then $0.114$ and a complex pair of size about 0.086.
$P^n$ computed as $X\Lambda^nX^{-1}$ (complex arithmetic, real result) equals NumPy's `matrix_power`. Already at $n = 5$
the four rows agree to four decimals — every starting base leads to the same distribution (0.2627, 0.2467, 0.2382,
0.2524), the stationary π of [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb), because the other eigenvalues have been raised to the 5th power
($0.114^5 \approx 2\times10^{-5}$). A chain "forgets" its starting base at the rate of its second-largest $|\lambda|$.

✏️ **Your turn.** For the poorer-habitat Leslie matrix of this section, how many years does it take until the growing mode makes up
99 % of the population? Use the closed form (the two other modes shrink relative to it by the factors
$\frac{1/2}{3/2} = \frac13$ and $\frac{1}{3/2} = \frac23$ per year — which one controls the answer?).

---
## 6 · The singular value decomposition

> 📍 **Where we are.** The eigendecomposition is limited: the matrix must be **square**, it needs $n$ independent
> eigenvectors, and its eigenvector matrix is usually **not orthogonal**. Data matrices are rectangular (24 samples ×
> 400 genes) and their rows and columns are different kinds of objects. The SVD removes all three limits: **every**
> matrix has one, and both of its outer factors are orthogonal.

**SVD theorem.** Let $A\in\mathbb R^{m\times n}$ be a rectangular matrix of rank $r\in[0, \min(m,n)]$. Then $A$ can be
written as

$$A = U\Sigma V^\top ,$$

where

* $U\in\mathbb R^{m\times m}$ and $V\in\mathbb R^{n\times n}$ are **orthogonal** matrices ($U^\top U = I_m$, $V^\top V = I_n$);
  their columns $\mathbf u_1,\dots,\mathbf u_m$ and $\mathbf v_1,\dots,\mathbf v_n$ are the **left-** and **right-singular
  vectors**;
* $\Sigma\in\mathbb R^{m\times n}$ has $\Sigma_{ii} = \sigma_i \ge 0$ and all other entries 0; the **singular values** are
  sorted, $\sigma_1\ge\sigma_2\ge\dots\ge\sigma_{\min(m,n)}\ge0$, and exactly $r$ of them are non-zero.

*In words:* every matrix is an orthogonal matrix times a "diagonal" matrix of non-negative numbers times another
orthogonal matrix; the number of non-zero singular values is the rank.

**The shape of $\Sigma$.** $\Sigma$ has the shape of $A$, so it is a diagonal matrix **padded with zero rows or
columns**:

$$m > n:\ \Sigma = \begin{pmatrix}\sigma_1 & & \\ & \ddots & \\ & & \sigma_n\\ 0 & \cdots & 0\\ \vdots & & \vdots\end{pmatrix},
\qquad m < n:\ \Sigma = \begin{pmatrix}\sigma_1 & & & 0 & \cdots\\ & \ddots & & \vdots & \\ & & \sigma_m & 0 & \cdots\end{pmatrix}.$$

*In words:* a tall $A$ gets extra zero rows below the diagonal block; a wide $A$ gets extra zero columns to its right.

**Outer-product form.** Multiplying out (only the first $r$ columns of $U$ and $V$ meet non-zero singular values):

$$A = \sum_{i=1}^{r}\sigma_i\,\mathbf u_i\mathbf v_i^\top, \qquad A\mathbf v_i = \sigma_i\mathbf u_i .$$

*In words:* $A$ is a sum of $r$ rank-one "layers" $\mathbf u_i\mathbf v_i^\top$ (an outer product: column × row), weighted by
the singular values; the right-singular vector $\mathbf v_i$ is sent to $\sigma_i$ times the left-singular vector
$\mathbf u_i$.

**The thin (economy) SVD.** The zero rows (or columns) of $\Sigma$ multiply columns of $U$ (or rows of $V^\top$) that
never contribute. Dropping them gives, for $m\ge n$, $U\in\mathbb R^{m\times n}$ (the shape of $A$), a **square**
$\Sigma\in\mathbb R^{n\times n}$ and $V\in\mathbb R^{n\times n}$ — "$U$ takes the shape of $A$ and $\Sigma$ is square". The
product is identical; the maths still works out.

⚠️ **Notation varies.** Some texts write $R = USV$ with $S$ for $\Sigma$ (because $\Sigma$ is reserved for sums or
covariance matrices) and **$V$ standing for what we call $V^\top$** — its *rows* are then the right-singular vectors.
Others write the thin SVD of an $n$ users × $m$ movies matrix as $E = FGH$ with $F\in\mathbb R^{n\times m}$,
$G = $ diagonal $m\times m$, $H\in\mathbb R^{m\times m}$ (our $U$, $\Sigma$, $V^\top$). Always check which factor is
transposed. NumPy, too, returns the transposed factor: `U, s, Vh = np.linalg.svd(A)` gives `Vh` $= V^\top$ — the right-singular
vectors are the **rows** of `Vh`, i.e. `Vh[i]`, not `Vh[:, i]`.

In [ ]:
# ── 6.1 np.linalg.svd: full vs thin shapes, padded Σ, rank = number of non-zero σ, the Vh trap ──
def padded_sigma(s, shape):
    """Build the m × n 'diagonal' Σ of a full SVD from the vector s of singular values (zero padding)."""
    S = np.zeros(shape)
    S[np.arange(len(s)), np.arange(len(s))] = s
    return S


A_tall = np.array([[1.0, -0.8], [0.0, 1.0], [1.0, 0.0]])          # 3 × 2 (m > n)
for M in [A_tall, A_tall.T]:                                         # tall and wide
    U_f, s, Vh = np.linalg.svd(M)                                    # full SVD
    Sigma = padded_sigma(s, M.shape)
    print(f"A is {M.shape[0]}×{M.shape[1]}: U {U_f.shape}, s = {s.round(4)}, Vh = Vᵀ {Vh.shape}; Σ =")
    print(format_matrix(Sigma.round(4)))
    print("   U Σ Vh == A ?", np.allclose(U_f @ Sigma @ Vh, M),
          "| UᵀU = I ?", np.allclose(U_f.T @ U_f, np.eye(M.shape[0])), "| VᵀV = I ?", np.allclose(Vh @ Vh.T, np.eye(M.shape[1])))

U_t, s_t, Vh_t = np.linalg.svd(A_tall, full_matrices=False)           # thin SVD
print("\nthin SVD of the 3×2 matrix: U", U_t.shape, " Σ", np.diag(s_t).shape, " Vᵀ", Vh_t.shape,
      "  U diag(s) Vᵀ == A ?", np.allclose(U_t @ np.diag(s_t) @ Vh_t, A_tall))
v1 = Vh_t[0]                                                          # first right-singular vector = first ROW of Vh
print("A v1 =", (A_tall @ v1).round(4), " σ1 u1 =", (s_t[0] * U_t[:, 0]).round(4))
print("wrong (a column of Vh):", (A_tall @ Vh_t[:, 0]).round(4))

rank1 = np.outer([1.0, 2.0, 3.0], [1.0, 0.5, -1.0, 2.0])              # rank 1 by construction
print("\nsingular values of a rank-1 3×4 matrix:", np.linalg.svd(rank1, compute_uv=False).round(6))

🔍 **What just happened?** For the $3\times2$ matrix the full SVD has $U$ of size $3\times3$, two singular values
$(1.6248, 1.0)$, $V^\top$ of size $2\times2$, and $\Sigma$ is $3\times2$ — the $2\times2$ diagonal block plus one row of
zeros. For its $2\times3$ transpose, $\Sigma$ is $2\times3$ with a zero **column** instead, and the singular values are
the same (the SVD of $A^\top$ is $V\Sigma^\top U^\top$). In both cases $U\Sigma V^\top$ reproduces the matrix and both
outer factors are orthogonal. The thin SVD drops the useless third column of $U$: $U$ is $3\times2$ (the shape of $A$)
and $\Sigma$ is a square $2\times2$. The first **row** of `Vh` satisfies $A\mathbf v_1 = \sigma_1\mathbf u_1$; the first
**column** does not — the most common SVD bug. Finally, a matrix built as one outer product has exactly one non-zero
singular value (the other two are 0 up to rounding): rank = number of non-zero singular values.

### 📘 Lecture notes — the SVD theorem

**Theorem.** $A\in\mathbb R^{m\times n}$ of rank $r\in[0,\min(m,n)]$ has a decomposition $A = U\Sigma V^\top$ with
$U\in\mathbb R^{m\times m}$, $V\in\mathbb R^{n\times n}$ orthogonal and $\Sigma\in\mathbb R^{m\times n}$ with $\Sigma_{ii} = \sigma_i$
and all other values 0; the singular values satisfy $\sigma_1\ge\sigma_2\ge\dots\ge\sigma_{\min(m,n)}\ge0$, and $r$ of them
are non-zero. The columns of $U$ ($\mathbf u_1,\dots,\mathbf u_m$) and of $V$ ($\mathbf v_1,\dots,\mathbf v_n$) are the
left- and right-singular vectors.

**Shapes.** $\Sigma = \widehat{\operatorname{diag}}_{m\times n}(\sigma_1,\dots)$: a diagonal matrix padded with 0 rows (when
$m > n$) or 0 columns (when $n > m$). Thin form: $U$ of the shape of $A$ and a square $\Sigma$ — equivalent.

**Relation to the eigendecomposition.** $A = X\Lambda X^{-1}$ needs a square, diagonalizable $A$ and uses one,
generally non-orthogonal, basis; $A = U\Sigma V^\top$ exists for every matrix and uses two orthonormal bases (one for
the input space $\mathbb R^n$, one for the output space $\mathbb R^m$), with non-negative "eigenvalue-like" numbers.

**Notation.** Some sources write $R = USV$ (with $S$ for $\Sigma$ and $V$ for our $V^\top$); `np.linalg.svd` returns
$V^\top$ as `Vh`. Rank-one expansion $A = \sum_{i\le r}\sigma_i\mathbf u_i\mathbf v_i^\top$, and $A\mathbf v_i = \sigma_i\mathbf u_i$.

> 🎓 **Exam tip.** Always state the shapes: for $A$ of size $m\times n$, $U$ is $m\times m$, $\Sigma$ is $m\times n$, $V^\top$ is
> $n\times n$ (full) — or $m\times k$, $k\times k$, $k\times n$ with $k = \min(m,n)$ (thin). The rank is the number of
> non-zero $\sigma_i$.

In [ ]:
# ── 6.2 The block picture of A = U Σ Vᵀ for a users × movies (or genes × assays) matrix, full and thin ──
def draw_block(ax, x, y, w, h, colour, label):
    """One coloured rectangle with a centred label (a matrix drawn to scale)."""
    ax.add_patch(plt.Rectangle((x, y - h), w, h, color=colour, ec="k", lw=0.5))
    ax.text(x + w / 2, y - h / 2, label, ha="center", va="center", fontsize=8, color="white", fontweight="bold")


m_, n_ = 8, 4                                        # 8 users (rows) × 4 movies (columns), drawn to scale
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, thin in zip(axes, [False, True]):
    k_ = n_ if thin else m_
    x0 = 0
    draw_block(ax, x0, m_, n_, m_, "#2196f3", f"A\n{m_}×{n_}"); x0 += n_ + 1.2
    ax.text(x0 - 0.6, m_ / 2, "=", fontsize=14, ha="center", va="center")
    draw_block(ax, x0, m_, k_, m_, "#f4b400", f"U\n{m_}×{k_}"); x0 += k_ + 0.3
    draw_block(ax, x0, m_, n_, k_, "#e53935", f"Σ\n{k_}×{n_}"); x0 += n_ + 0.3
    draw_block(ax, x0, m_, n_, n_, "#43a047", f"Vᵀ\n{n_}×{n_}"); x0 += n_
    ax.text(-0.4, m_ / 2, "users", rotation=90, ha="center", va="center", fontsize=8)
    ax.text(n_ / 2, m_ + 0.4, "movies", ha="center", fontsize=8)
    ax.set_xlim(-1, x0 + 0.5); ax.set_ylim(-0.5, m_ + 1); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("thin SVD (U takes the shape of A, Σ square)" if thin else "full SVD (Σ padded with zero rows)",
                 fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The blocks are drawn to scale for 8 users × 4 movies. In the **full** SVD, $U$ is a square
$8\times8$ and $\Sigma$ is $8\times4$, of which only the top $4\times4$ block can be non-zero; in the **thin** SVD, $U$ is
$8\times4$ — the shape of $A$ — and $\Sigma$ is a square $4\times4$. The same picture describes a genes × assays
expression matrix (rows = genes, columns = assays). Reading it as a recipe: each user (row of $U$) has a few "taste"
coordinates, each movie (column of $V^\top$) a few "genre" coordinates, and $\Sigma$ says how strongly each taste dimension
connects users to movies. Section 9 makes this concrete.

---
## 7 · SVD intuition: rotate, stretch (and change dimension), rotate

> 📍 **Where we are.** As with $PDP^{-1}$, read $A = U\Sigma V^\top$ from right to left: $V^\top$ is orthogonal — a
> rotation (possibly combined with a reflection) **in the input space** $\mathbb R^n$ — that turns the right-singular
> vectors $\mathbf v_i$ onto the coordinate axes $\mathbf e_i$ (because $V^\top\mathbf v_i = \mathbf e_i$); $\Sigma$
> stretches axis $i$ by $\sigma_i$ and **changes the dimension** from $n$ to $m$ (adding or dropping zero coordinates);
> $U$ rotates **in the output space** $\mathbb R^m$, sending $\mathbf e_i$ to $\mathbf u_i$. Consequence: **every matrix
> maps the unit circle (sphere) to an ellipse (ellipsoid)** whose semi-axes are $\sigma_i\mathbf u_i$.

We follow the classic example (Deisenroth, Faisal & Ong) of a map from $\mathbb R^2$ to $\mathbb R^3$:

$$A = \begin{pmatrix}1 & -0.8\\ 0 & 1\\ 1 & 0\end{pmatrix} = U\Sigma V^\top \approx
\begin{pmatrix}-0.79 & 0 & -0.62\\ 0.38 & -0.78 & -0.49\\ -0.48 & -0.62 & 0.62\end{pmatrix}
\begin{pmatrix}1.62 & 0\\ 0 & 1.0\\ 0 & 0\end{pmatrix}
\begin{pmatrix}-0.78 & 0.62\\ -0.62 & -0.78\end{pmatrix}.$$

*In words:* the $2\times2$ factor $V^\top$ rotates the plane, $\Sigma$ stretches by 1.62 and 1.0 and places the plane into
3-D space (third coordinate 0), and the $3\times3$ factor $U$ tilts that plane into its final position.

In [ ]:
# ── 7.1 The four stages of A = U Σ Vᵀ for a 3 × 2 matrix: 400 colour-coded points and the unit circle ──
U_a, s_a, Vh_a = np.linalg.svd(A_tall)               # full SVD: U 3×3, s (2,), Vh 2×2
Sig_a = padded_sigma(s_a, A_tall.shape)
print("U =\n", U_a.round(2), "\nσ =", s_a.round(3), "\nVᵀ =\n", Vh_a.round(2))
print("det Vᵀ =", round(np.linalg.det(Vh_a), 3), "(+1: a rotation)  | det U =", round(np.linalg.det(U_a), 3))

g = np.linspace(-1, 1, 20)
gx, gy = np.meshgrid(g, g)
grid = np.vstack([gx.ravel(), gy.ravel()])
colours = plt.cm.viridis((grid[1] + 1) / 2)
circle2 = np.vstack([np.cos(phi), np.sin(phi)])
stages = [grid, Vh_a @ grid, Sig_a @ Vh_a @ grid, A_tall @ grid]
circles = [circle2, Vh_a @ circle2, Sig_a @ Vh_a @ circle2, A_tall @ circle2]
v_cols = [Vh_a[0], Vh_a[1]]                                           # right-singular vectors v1, v2
arrows = [v_cols, [Vh_a @ v for v in v_cols], [Sig_a @ Vh_a @ v for v in v_cols], [A_tall @ v for v in v_cols]]
titles = ["x and v1, v2 (in ℝ²)", "Vᵀx: v_i → e_i (rotation)", "ΣVᵀx: stretch σ1, σ2; now in ℝ³",
          "UΣVᵀx = Ax: σ_i e_i → σ_i u_i"]

fig = plt.figure(figsize=(13, 4.0))
for k in range(4):
    is3d = k >= 2
    ax = fig.add_subplot(1, 4, k + 1, projection="3d" if is3d else None)
    pts, circ = stages[k], circles[k]
    if is3d:
        ax.scatter(pts[0], pts[1], pts[2], c=colours, s=2, alpha=0.35, depthshade=False)
        ax.plot(circ[0], circ[1], circ[2], color="k", lw=2)
        for vec, c in zip(arrows[k], ["tab:red", "tab:orange"]):
            ax.quiver(0, 0, 0, *vec, color=c, lw=3, arrow_length_ratio=0.2)
        ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_zlim(-1.4, 1.4)
        ax.set_xlabel("$x_1$", fontsize=8); ax.set_ylabel("$x_2$", fontsize=8); ax.set_zlabel("$x_3$", fontsize=8)
        ax.tick_params(labelsize=6); ax.view_init(elev=22, azim=-60)
    else:
        ax.scatter(pts[0], pts[1], c=colours, s=4)
        ax.plot(circ[0], circ[1], color="k", lw=1)
        plot_vectors(arrows[k], colors=["tab:red", "tab:orange"], ax=ax, lim=(-1.6, 1.6))
        ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    ax.set_title(titles[k], fontsize=9)
fig.suptitle("A = U Σ Vᵀ maps the unit circle (black) to an ellipse; red/orange = images of v1/v2", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** NumPy reproduces the factors of the textbook example: $\sigma = (1.625, 1.0)$,
$U$ and $V^\top$ with the printed entries (the signs match the textbook here; in general each pair
$\mathbf u_i, \mathbf v_i$ may come out with both signs flipped). $\det V^\top = +1$ and $\det U = +1$, so here both outer
steps are pure rotations (in general an orthogonal matrix may also include a reflection, $\det = -1$).

Follow the panels: (1) the 400 points fill a square, the unit circle is drawn in black and the right-singular vectors
$\mathbf v_1$ (red) and $\mathbf v_2$ (orange) are two perpendicular unit vectors; (2) $V^\top$ rotates them onto the
coordinate axes, carrying the square along (now tilted); (3) $\Sigma$ stretches the first axis by 1.62 and keeps the
second ($\sigma_2 = 1$), and embeds the plane in 3-D as the flat sheet $x_3 = 0$ — the circle has become an
axis-aligned ellipse; (4) $U$ (a rotation, as $\det U = +1$) turns the sheet into a tilted plane in $\mathbb R^3$, and the ellipse's
semi-axes become $\sigma_1\mathbf u_1$ and $\sigma_2\mathbf u_2$. The image of the whole plane $\mathbb R^2$ is a
2-dimensional plane inside $\mathbb R^3$ — rank 2 — and the third left-singular vector $\mathbf u_3$, which no input
reaches, is perpendicular to it.

---
## 8 · Constructing the SVD by hand

> 📍 **Where we are.** Where do $U$, $\Sigma$ and $V$ come from? From the eigendecomposition of a **symmetric positive
> semidefinite** matrix — and [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) showed that $A^\top A$ is one, for *any* $A$.

**Starting point.** For a symmetric PSD matrix $S$, the spectral theorem gives $S = S^\top = PDP^\top$ with orthogonal
$P$ and non-negative eigenvalues in $D$. An SVD of the same $S$ would be $S = U\Sigma V^\top$; choosing $U = V = P$ and
$\Sigma = D$ shows that **for symmetric PSD matrices the eigendecomposition is an SVD**. For a general $A$ we use the PSD
matrix $A^\top A$.

**Step 1 — right-singular vectors and singular values from $A^\top A$.** If $A = U\Sigma V^\top$, then

$$A^\top A = (U\Sigma V^\top)^\top(U\Sigma V^\top) = V\Sigma^\top U^\top U\Sigma V^\top = V\,(\Sigma^\top\Sigma)\,V^\top .$$

*In words:* transpose the product (order reverses), and $U^\top U = I$ cancels in the middle. $\Sigma^\top\Sigma$ is an
$n\times n$ diagonal matrix with entries $\sigma_i^2$. This *is* an eigendecomposition of the symmetric matrix
$A^\top A$, so: **the right-singular vectors $\mathbf v_i$ are orthonormal eigenvectors of $A^\top A$, and
$\sigma_i = \sqrt{\lambda_i}$** where $\lambda_i\ge0$ are its eigenvalues (sorted descending). (Similarly
$AA^\top = U(\Sigma\Sigma^\top)U^\top$: the left-singular vectors are eigenvectors of $AA^\top$, with the same non-zero
eigenvalues.)

**Step 2 — the singular-value matrix.** Put $\sigma_1\ge\sigma_2\ge\dots$ on the diagonal of an $m\times n$ zero matrix.

**Step 3 — left-singular vectors as normalised images.** We need an orthonormal $U$ paired with $V$ such that
$A\mathbf v_i = \sigma_i\mathbf u_i$. So define, for every $\sigma_i > 0$,

$$\mathbf u_i = \frac{A\mathbf v_i}{\|A\mathbf v_i\|} = \frac{1}{\sqrt{\lambda_i}}A\mathbf v_i = \frac{1}{\sigma_i}A\mathbf v_i ,
\qquad i = 1,\dots,r .$$

*In words:* push each right-singular vector through $A$ and rescale it to length 1. The rescaling factor is exactly
$\sigma_i$, because

$$\|A\mathbf v_i\|^2 = (A\mathbf v_i)^\top(A\mathbf v_i) = \mathbf v_i^\top(A^\top A)\mathbf v_i = \mathbf v_i^\top(\lambda_i\mathbf v_i)
= \lambda_i(\mathbf v_i^\top\mathbf v_i) = \lambda_i .$$

*In words:* the squared length of $A\mathbf v_i$ is the eigenvalue $\lambda_i$ (using $A^\top A\mathbf v_i = \lambda_i\mathbf v_i$ and
$\mathbf v_i^\top\mathbf v_i = 1$), so $\|A\mathbf v_i\| = \sqrt{\lambda_i} = \sigma_i$. The same calculation with two different
indices, $(A\mathbf v_i)^\top(A\mathbf v_j) = \lambda_j\mathbf v_i^\top\mathbf v_j = 0$, shows that the $\mathbf u_i$ are mutually
orthogonal. If $r < m$, complete $\mathbf u_{r+1},\dots,\mathbf u_m$ with any orthonormal basis of the remaining directions.

### 8.1 Worked example: find the SVD of $A = \begin{pmatrix}1&0&1\\-2&1&0\end{pmatrix}$

**Step 1.** $A^\top A = \begin{pmatrix}1&-2\\0&1\\1&0\end{pmatrix}\begin{pmatrix}1&0&1\\-2&1&0\end{pmatrix}
= \begin{pmatrix}5&-2&1\\-2&1&0\\1&0&1\end{pmatrix}$ (entry $(i,j)$ = dot product of columns $i$ and $j$ of $A$, e.g.
$(1,1)$: $1 + 4 = 5$). Its characteristic polynomial is $-\lambda(\lambda-6)(\lambda-1)$, so the eigenvalues are 6, 1, 0, with
unit eigenvectors

$$\mathbf v_1 = \tfrac{1}{\sqrt{30}}(5, -2, 1),\quad \mathbf v_2 = \tfrac{1}{\sqrt5}(0, 1, 2),\quad \mathbf v_3 = \tfrac{1}{\sqrt6}(-1,-2,1),
\qquad V = \begin{pmatrix}\frac{5}{\sqrt{30}} & 0 & \frac{-1}{\sqrt6}\\ \frac{-2}{\sqrt{30}} & \frac{1}{\sqrt5} & \frac{-2}{\sqrt6}\\
\frac{1}{\sqrt{30}} & \frac{2}{\sqrt5} & \frac{1}{\sqrt6}\end{pmatrix}.$$

*In words:* $A^\top A = VDV^\top$ with $D = \operatorname{diag}(6, 1, 0)$; the columns of $V$ are the right-singular vectors.

**Step 2.** $\operatorname{rank}(A) = 2$, so there are two non-zero singular values, $\sigma_1 = \sqrt6$ and $\sigma_2 = 1$, and
$\Sigma$ has the shape of $A$:

$$\Sigma = \begin{pmatrix}\sqrt6 & 0 & 0\\ 0 & 1 & 0\end{pmatrix}.$$

*In words:* the two non-zero singular values sit on the diagonal of a $2\times3$ matrix; the zero third column belongs to
the input direction $\mathbf v_3$ that $A$ sends to $\mathbf 0$.

**Step 3.**

$$\mathbf u_1 = \frac{1}{\sigma_1}A\mathbf v_1 = \frac{1}{\sqrt6}\cdot\frac{1}{\sqrt{30}}\begin{pmatrix}5 + 1\\ -10 - 2\end{pmatrix}
= \frac{1}{\sqrt{180}}\begin{pmatrix}6\\-12\end{pmatrix} = \begin{pmatrix}\frac{1}{\sqrt5}\\ -\frac{2}{\sqrt5}\end{pmatrix},\qquad
\mathbf u_2 = \frac{1}{\sigma_2}A\mathbf v_2 = \frac{1}{\sqrt5}\begin{pmatrix}0 + 2\\ 0 + 1\end{pmatrix} = \begin{pmatrix}\frac{2}{\sqrt5}\\ \frac{1}{\sqrt5}\end{pmatrix},$$

$$U = [\mathbf u_1, \mathbf u_2] = \frac{1}{\sqrt5}\begin{pmatrix}1 & 2\\ -2 & 1\end{pmatrix}.$$

*In words:* $\sqrt{180} = 6\sqrt5$, so $\mathbf u_1 = (1, -2)/\sqrt5$; $\mathbf u_2 = (2,1)/\sqrt5$ is perpendicular to it.
Since $m = 2 = r$, $U$ is already complete. ($\mathbf v_3$ spans the null space of $A$: $A\mathbf v_3 = \mathbf 0$.)

In [ ]:
# ── 8.2 The worked example step by step with SymPy (exact), then np.linalg.svd ──
A_ex = sp.Matrix([[1, 0, 1], [-2, 1, 0]])
AtA = A_ex.T * A_ex
print("step 1: AᵀA =", AtA.tolist(), "  det(AᵀA − λI) =", sp.factor((AtA - lam * sp.eye(3)).det()))
eig = sorted(AtA.eigenvects(), key=lambda e: -e[0])               # eigenvalues 6, 1, 0 (descending)
V_ex = sp.Matrix.hstack(*[ev[2][0] / ev[2][0].norm() for ev in eig])
print("        V =", [[sp.nsimplify(x) for x in row] for row in V_ex.tolist()])
sig = [sp.sqrt(ev[0]) for ev in eig]
print("step 2: singular values σ = √λ =", sig)
Sig_ex = sp.zeros(2, 3)
Sig_ex[0, 0], Sig_ex[1, 1] = sig[0], sig[1]
U_ex = sp.Matrix.hstack(*[sp.simplify(A_ex * V_ex[:, i] / sig[i]) for i in range(2)])
print("step 3: U =", U_ex.tolist())
print("check: U Σ Vᵀ == A ?", sp.simplify(U_ex * Sig_ex * V_ex.T) == A_ex,
      "| UᵀU = I ?", sp.simplify(U_ex.T * U_ex) == sp.eye(2), "| A v3 =", list(A_ex * V_ex[:, 2]))
print("AAᵀ =", (A_ex * A_ex.T).tolist(), "with eigenvalues", (A_ex * A_ex.T).eigenvals(), "(same non-zero λ)")

U_np, s_np, Vh_np = np.linalg.svd(np.array([[1.0, 0, 1], [-2, 1, 0]]))
print("\nNumPy: σ =", s_np.round(6), " (√6 =", round(np.sqrt(6), 6), ")")
print("U =\n", U_np.round(4), "\nVᵀ =\n", Vh_np.round(4) + 0.0)
print("by hand: u1 =", np.array(U_ex[:, 0], dtype=float).ravel().round(4), " v1 =", np.array(V_ex[:, 0], dtype=float).ravel().round(4))

🔍 **What just happened?** SymPy reproduces every step exactly: $A^\top A$ has eigenvalues 6, 1, 0; the singular
values are $\sqrt6\approx2.4495$ and 1; $U = \frac{1}{\sqrt5}\begin{pmatrix}1&2\\-2&1\end{pmatrix}$; and $U\Sigma V^\top = A$,
$U^\top U = I$, $A\mathbf v_3 = \mathbf 0$. $AA^\top = \begin{pmatrix}2&-2\\-2&5\end{pmatrix}$ has the same non-zero eigenvalues 6
and 1. NumPy finds the same singular values, but its first left- and right-singular vectors are the **negatives** of
ours ($\mathbf u_1 = (-0.4472, 0.8944)$, $\mathbf v_1 = -(0.9129, -0.3651, 0.1826)$). That is not a disagreement: flipping
the signs of $\mathbf u_i$ and $\mathbf v_i$ together leaves $\sigma_i\mathbf u_i\mathbf v_i^\top$ unchanged. Its third row of
$V^\top$ is $(-1,-2,1)/\sqrt6 = (-0.4082, -0.8165, 0.4082)$, our $\mathbf v_3$.

⚠️ **Common confusion — computing $U$ independently.** If you take the eigenvectors of $AA^\top$ for $U$ and those of
$A^\top A$ for $V$ separately, their signs may not match, and $U\Sigma V^\top$ comes out wrong. Compute
$\mathbf u_i = A\mathbf v_i/\sigma_i$ from the $\mathbf v_i$ — then the pairing is automatic.

**SVD vs eigendecomposition for symmetric matrices.** If $S$ is symmetric PSD, $S = PDP^\top$ is already an SVD
($U = V = P$, $\Sigma = D$). If $S$ is symmetric but has negative eigenvalues, the singular values are $|\lambda_i|$ and
the signs move into $U$: e.g. $\begin{pmatrix}1&2\\2&1\end{pmatrix}$ has eigenvalues 3 and $-1$, singular values 3 and 1.

✏️ **Your turn.** Find the SVD of $\begin{pmatrix}3&0\\4&5\end{pmatrix}$ by hand ($A^\top A = \begin{pmatrix}25&20\\20&25\end{pmatrix}$, eigenvalues
45 and 5) and check with `np.linalg.svd`. Then verify numerically that the singular values of
$\begin{pmatrix}1&2\\2&1\end{pmatrix}$ are the absolute values of its eigenvalues.

### 📘 Lecture notes — constructing the SVD

**From symmetric PSD to general.** For symmetric positive semidefinite $S = S^\top = PDP^\top$, the SVD $S = U\Sigma V^\top$
is the same factorisation ($V = U = P$, $\Sigma = D$). Any matrix gives a square PSD matrix $A^\top A$ ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)).

**Recipe.** (1) Eigendecompose $A^\top A = V\Sigma^\top\Sigma V^\top$: the right-singular vectors are its orthonormal
eigenvectors, and $\sigma_i = \sqrt{\lambda_i}$, sorted descending. (2) $\Sigma$ = $m\times n$ with $\sigma_i$ on the diagonal.
(3) $\mathbf u_i = A\mathbf v_i/\|A\mathbf v_i\| = A\mathbf v_i/\sqrt{\lambda_i} = A\mathbf v_i/\sigma_i \iff A\mathbf v_i = \sigma_i\mathbf u_i$,
$i = 1,\dots,\min(m,n)$ with $\sigma_i > 0$, because $(A\mathbf v_i)^\top(A\mathbf v_i) = \mathbf v_i^\top A^\top A\mathbf v_i = \lambda_i\mathbf v_i^\top\mathbf v_i
= \lambda_i$; complete $U$ to an orthonormal basis if needed.

**Worked example.** $A = \begin{pmatrix}1&0&1\\-2&1&0\end{pmatrix}$: $A^\top A = \begin{pmatrix}5&-2&1\\-2&1&0\\1&0&1\end{pmatrix}$,
$\lambda = 6, 1, 0$; $\mathbf v_1 = (5,-2,1)/\sqrt{30}$, $\mathbf v_2 = (0,1,2)/\sqrt5$, $\mathbf v_3 = (-1,-2,1)/\sqrt6$;
$\Sigma = \begin{pmatrix}\sqrt6&0&0\\0&1&0\end{pmatrix}$ (rank 2); $\mathbf u_1 = (1,-2)/\sqrt5$, $\mathbf u_2 = (2,1)/\sqrt5$,
$U = \frac{1}{\sqrt5}\begin{pmatrix}1&2\\-2&1\end{pmatrix}$.

**Common confusions.** Singular values are the square roots of the eigenvalues of $A^\top A$, not the eigenvalues of $A$
(for symmetric $A$ they are $|\lambda_i|$). Signs of singular-vector pairs are arbitrary but must be flipped together.

> 🎓 **Exam tip.** Use the smaller of $A^\top A$ ($n\times n$) and $AA^\top$ ($m\times m$) to find the non-zero $\sigma_i$ —
> they have the same non-zero eigenvalues — then get the other singular vectors from $\mathbf u_i = A\mathbf v_i/\sigma_i$
> (or $\mathbf v_i = A^\top\mathbf u_i/\sigma_i$). Check $\sum\sigma_i^2 = \sum_{ij}a_{ij}^2$ ($6 + 1 = 7 = 1 + 1 + 4 + 1$).

---
## 9 · Low-rank approximation: keep the big singular values

> 📍 **Where we are.** The outer-product form $A = \sum_i\sigma_i\mathbf u_i\mathbf v_i^\top$ orders the "layers" of a matrix
> by importance. Keeping only the first $k$ layers gives the best possible approximation of rank $k$ — the basis of
> denoising, compression and the prediction of missing entries.

**Truncated SVD.** For $k < r$,

$$A_k = \sum_{i=1}^{k}\sigma_i\,\mathbf u_i\mathbf v_i^\top = U_k\Sigma_kV_k^\top \qquad(\operatorname{rank}A_k = k).$$

*In words:* keep the first $k$ columns of $U$ and $V$ and the $k$ largest singular values; drop the rest.

🧮 **Math refresher — the Frobenius norm.** The size of a whole matrix is measured by
$\|M\|_F = \sqrt{\sum_{i,j}M_{ij}^2}$ — the Euclidean length of the matrix seen as one long vector (Module 07). Because
orthogonal matrices do not change lengths, $\|A\|_F^2 = \sum_i\sigma_i^2$.

**Eckart–Young theorem (stated without proof).** Among **all** matrices $B$ of rank at most $k$, the truncated SVD is
the closest to $A$:

$$\min_{\operatorname{rank}B\le k}\|A - B\|_F = \|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \dots + \sigma_r^2}.$$

*In words:* the squared reconstruction error of the best rank-$k$ approximation is exactly the sum of the squared
singular values you threw away; the fraction of $\|A\|_F^2$ that is kept is $\sum_{i\le k}\sigma_i^2/\sum_i\sigma_i^2$.

### 9.1 Users × movies: what do $U$, $\Sigma$ and $V$ mean?

The classic recommender example: a matrix with one row per **user** and one column per **movie**, each entry the
user's rating (1–5). Three users love science fiction, two love romance, and one (user 6) likes both and has not seen
*Star Wars* (illustrative, typed-in numbers).

In [ ]:
# ── 9.1 A small ratings matrix: rank-2 structure, interpretation, and a predicted missing rating (illustrative) ──
movies = ["Alien", "Star Wars", "Matrix", "Casablanca", "Titanic"]
ratings = pd.DataFrame([[5, 5, 4, 1, 2], [4, 5, 5, 2, 1], [5, 4, 5, 1, 1], [1, 2, 1, 5, 4], [2, 1, 1, 4, 5],
                        [4, np.nan, 5, 2, 4]], columns=movies, index=[f"user {i}" for i in range(1, 7)])
print(ratings)


def svd_impute(R, k, n_iter=500, tol=1e-10):
    """Fill the NaNs of R with a rank-k approximation: start from column means, then repeatedly replace the missing
    entries by the rank-k truncated SVD of the completed matrix (the observed entries stay fixed)."""
    R = np.asarray(R, dtype=float)
    missing = np.isnan(R)
    F = np.where(missing, np.nanmean(R, axis=0), R)
    for it in range(n_iter):
        U_, s_, Vh_ = np.linalg.svd(F, full_matrices=False)
        Ak = U_[:, :k] * s_[:k] @ Vh_[:k]
        F_new = np.where(missing, Ak, R)
        if np.max(np.abs(F_new - F)) < tol:
            break
        F = F_new
    return F_new, it + 1


filled, n_it = svd_impute(ratings.to_numpy(), k=2)
print(f"\npredicted rating of user 6 for Star Wars (rank 2, {n_it} iterations): {filled[5, 1]:.2f}")
U_r, s_r, Vh_r = np.linalg.svd(filled, full_matrices=False)
print("singular values:", s_r.round(2), "→ share of Σσ² in the first two:", round(float(np.sum(s_r[:2] ** 2) / np.sum(s_r ** 2)), 4))
print("\nV (movies × taste dimensions, first two):")
print(pd.DataFrame(Vh_r[:2].T, index=movies, columns=["v1", "v2"]).round(2).T.to_string())
print("\nU (users × taste dimensions, first two):")
print(pd.DataFrame(U_r[:, :2], index=ratings.index, columns=["u1", "u2"]).round(2).T.to_string())

🔍 **What just happened?** The completed matrix has singular values $(18.02, 7.43, 2.01, 1.31, 1.10)$: the first two
carry 98 % of $\sum\sigma_i^2$, so the ratings are close to rank 2. The singular vectors are interpretable:

* $\mathbf v_1$ has the same sign for every movie (all about $-0.3$ to $-0.5$) — "how much a person likes movies at all";
  $\mathbf u_1$ similarly gives every user a similar weight.
* $\mathbf v_2$ separates the genres: positive for *Alien*, *Star Wars*, *Matrix* (0.22–0.32), negative for *Casablanca* and
  *Titanic* ($-0.62$, $-0.63$) — a **sci-fi vs romance** axis. $\mathbf u_2$ places users 1–3 on the sci-fi side (0.23–0.33),
  users 4–5 on the romance side ($-0.63$), and user 6 in the middle ($-0.02$): a fan of both.
* $\Sigma$ weights the two "taste dimensions": 18.0 for general liking, 7.4 for the genre contrast.

This is the meaning of the classic description "$U$ and $V$ show commonalities between users and between movies, and
$\Sigma$ is a connection matrix". The rank-2 model predicts that user 6 would rate *Star Wars* about **4.7**:
someone who rated *Alien* 4 and *Matrix* 5 probably likes *Star Wars* too. (With only 6 users this is a toy; real
recommenders use millions of users and regularised versions of the same idea.)

### 9.2 Denoising: how many programmes are in the data?

Now a **SIMULATED** biological matrix with a **known truth**: 30 samples × 200 genes generated by **3 regulatory programmes** (each
sample has an activity for each programme, each gene a loading on each programme — a rank-3 matrix), plus independent
measurement noise.

In [ ]:
# ── 9.2 Denoising by truncation: error against the known rank-3 truth (SIMULATED) ──
rng_d = np.random.default_rng(SEED + 1)
N_d, J_d = 30, 200
activities = rng_d.normal(size=(N_d, 3))                              # programme activity in each sample
loadings = rng_d.normal(size=(3, J_d)) * np.array([[3.0], [2.0], [1.2]])  # gene loadings, programmes of decreasing size
truth = activities @ loadings                                          # rank 3 by construction
noisy = truth + rng_d.normal(0, 1.0, size=(N_d, J_d))                 # measurement noise, SD 1

U_n, s_n, Vh_n = np.linalg.svd(noisy, full_matrices=False)
rows = []
for k in range(1, 9):
    A_k = U_n[:, :k] * s_n[:k] @ Vh_n[:k]
    rows.append(dict(k=k, err_vs_truth=np.linalg.norm(A_k - truth) / np.linalg.norm(truth),
                     residual_sq=np.linalg.norm(noisy - A_k) ** 2, discarded_sigma_sq=np.sum(s_n[k:] ** 2)))
table = pd.DataFrame(rows).set_index("k")
print("rank of the truth:", np.linalg.matrix_rank(truth), "| rank of the noisy data:", np.linalg.matrix_rank(noisy))
print(f"error of the raw noisy data vs truth: {np.linalg.norm(noisy - truth) / np.linalg.norm(truth):.3f}")
print(table.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(range(1, 16), s_n[:15], "o-")
axes[0].set_xlabel("index i"); axes[0].set_ylabel("singular value σ_i"); axes[0].set_title("scree plot: 3 programmes, then a noise floor")
axes[1].plot(table.index, table["err_vs_truth"], "o-", label="rank-k approximation")
axes[1].axhline(np.linalg.norm(noisy - truth) / np.linalg.norm(truth), color="0.5", ls="--", label="raw noisy data")
axes[1].set_xlabel("k (number of singular values kept)"); axes[1].set_ylabel("relative error vs truth")
axes[1].set_title("denoising works best at the true rank"); axes[1].legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The truth has rank 3, but noise makes the observed matrix full rank (30). The scree plot
shows **three large singular values** standing above a flat **noise floor** — the signature of three programmes. The
table confirms Eckart–Young: the squared residual $\|A - A_k\|_F^2$ equals the sum of the discarded $\sigma_i^2$ in
every row. More importantly, compare with the truth: the raw data are off by about 31 % (relative Frobenius error); the
rank-3 approximation is off by only about 10 % — truncation removed most of the noise, because noise spreads its
energy over all 30 directions while the signal lives in 3. Keeping fewer than 3 throws away signal (56 % and 28 %
error for $k = 1, 2$); keeping more than 3 lets noise back in (the error rises again for $k = 4, 5, \dots$). Choosing $k$
from the elbow of the scree plot is the everyday version of this experiment.

### 🌉 Bridge — filling in a drug × cell-line response matrix

Pharmacogenomic screens test many drugs on many cancer cell lines, and the response matrix is never complete. Because
drug responses are driven by a few shared factors (a baseline sensitivity, a proliferation rate, dependence on a
pathway), the matrix is approximately **low rank**, and a missing response can be predicted from the others exactly as
user 6's rating was. The cell below simulates 24 cell lines × 10 drugs from a rank-3 truth, hides entries at random (each with probability 0.15),
and compares SVD completion with the naive "column mean" guess. The same idea — matrix completion — is used for
missing genotypes and for sparse single-cell data; the 02-620 companion meets the same low-rank idea in
[BiologyToML · Module 14 — PCA](../../../../ml/tutorial/BiologyToML/notebooks/14_pca_dimensionality_reduction.ipynb) and its non-linear, probabilistic cousin in
[Module 25 — variational autoencoders](../../../../ml/tutorial/BiologyToML/notebooks/25_variational_inference_vae.ipynb).

In [ ]:
# ── 9.3 Matrix completion of a SIMULATED drug × cell-line response matrix (rank-3 truth, each entry hidden with probability 0.15) ──
rng_c = np.random.default_rng(SEED + 3)
n_cell, n_drug = 24, 10
response_truth = 0.5 + 0.8 * rng_c.normal(size=(n_cell, 2)) @ rng_c.normal(size=(2, n_drug))   # baseline + 2 factors
observed = response_truth + rng_c.normal(0, 0.3, size=response_truth.shape)                  # assay noise
hidden = rng_c.random(response_truth.shape) < 0.15
R_obs = np.where(hidden, np.nan, observed)
print(f"{hidden.sum()} of {hidden.size} responses hidden; rank of the noise-free truth: {np.linalg.matrix_rank(response_truth)}")

col_mean_guess = np.nanmean(R_obs, axis=0)[np.where(hidden)[1]]
print(f"column-mean guess: RMSE vs truth on the hidden entries = {np.sqrt(np.mean((col_mean_guess - response_truth[hidden]) ** 2)):.3f}")
for k in [1, 2, 3, 4]:
    completed, n_it = svd_impute(R_obs, k=k, n_iter=2000)
    rmse = np.sqrt(np.mean((completed[hidden] - response_truth[hidden]) ** 2))
    print(f"rank-{k} SVD completion ({n_it:4d} iterations): RMSE = {rmse:.3f}")

🔍 **What just happened?** 46 of the 240 responses were hidden. Guessing each with its drug's average misses the truth
by an RMSE of about 1.5; rank-3 SVD completion — matching the true structure (a constant baseline is itself a rank-one
layer, plus two factors) — brings it down to about 0.28, close to the assay noise of 0.3. Too small a rank (1 or 2)
cannot represent the structure; too large a rank (4) starts fitting noise: the error rises again (about 0.74) and the iteration does not even settle
within 2,000 steps. As in
denoising, the rank is a modelling choice that must be validated — here against hidden entries, in practice by
cross-validation.

### 📘 Lecture notes — low-rank approximation and applications

**Truncation.** $A_k = \sum_{i\le k}\sigma_i\mathbf u_i\mathbf v_i^\top$; Eckart–Young: $A_k$ is the best rank-$k$ approximation,
with $\|A - A_k\|_F^2 = \sum_{i>k}\sigma_i^2$; fraction kept $\sum_{i\le k}\sigma_i^2/\sum_i\sigma_i^2$.

**Movie prediction.** $A\in\mathbb R^{m\times n}$: each of the $m$ **rows** is a user, each of the $n$ **columns** a movie,
entries are ratings. $A = U\Sigma V^\top$: $U$ ($m\times m$) relates users to latent "taste" dimensions, $V$ ($n\times n$)
relates movies to the same dimensions, $\Sigma$ ($m\times n$) weights the dimensions. In the thin form $E = FGH$ for
$n$ users × $m$ movies, $F\in\mathbb R^{n\times m}$, $G$ diagonal $m\times m$, $H = V^\top\in\mathbb R^{m\times m}$. Missing
ratings are predicted from a low-rank approximation.

**Gene expression.** $A\in\mathbb R^{m\times n}$: each of the $m$ **rows** is a gene, each of the $n$ **columns** an assay
(array, condition, sample). The left-singular vectors (columns of $U$, one entry per gene) are eigenvectors of the
gene × gene matrix $AA^\top$; the right-singular vectors (columns of $V$, one entry per assay) are eigenvectors of the
assay × assay matrix $A^\top A$ — after centring/standardising, these are proportional to the gene–gene and
assay–assay covariance (correlation) matrices; $\Sigma$ links the two ("a correlation matrix of genes, a correlation
of assays and a relationship matrix"). Alter *et al.* (2000) call the assay patterns **eigengenes** and the gene
patterns **eigenarrays**.

**Common confusions.** Rows vs columns (users and genes are rows here; a samples × genes matrix is the transpose, so
the roles of $U$ and $V$ swap). Low-rank structure is an assumption to be checked (scree plot, held-out entries).

> 🎓 **Exam tip.** Given singular values, compute the share of $\|A\|_F^2$ kept by the top $k$ and the error
> $\sqrt{\sum_{i>k}\sigma_i^2}$; explain in words what the first one or two singular vector pairs represent.

---
## 10 · Eigengenes of the infection experiment: SVD = PCA

> 📍 **Where we are.** Back to the question that opened the module. We apply the SVD to the 24 × 400 expression matrix
> and read off the patterns.

**Orientation.** The gene-expression application above writes genes as rows and assays as columns. The course
convention puts **samples in rows** ($X$: $N = 24$ samples × $J = 400$ genes), i.e. the transpose — so the roles swap:
in $X_c = U\Sigma V^\top$, a column of $U$ has one entry per **sample** (an expression pattern across assays: an
**eigengene** in Alter's terms) and a column of $V$ has one entry per **gene** (a pattern across genes: an
**eigenarray**, i.e. the gene loadings).

**Centre first, and you have PCA.** Subtract each gene's mean to get $X_c$. Then

$$S = \frac{1}{N-1}X_c^\top X_c = V\,\frac{\Sigma^\top\Sigma}{N-1}\,V^\top ,$$

*In words:* the covariance matrix of the genes (Module 12's covariance, now $400\times400$) has the right-singular vectors
$\mathbf v_i$ as eigenvectors — the **principal directions** — and the eigenvalues $\sigma_i^2/(N-1)$ — the **variances**
along them. The **principal component scores** of the samples are $X_cV = U\Sigma$, and the fraction of variance explained
by component $i$ is $\sigma_i^2/\sum_j\sigma_j^2$ (Alter's "fraction of eigenexpression"). PCA is the SVD of the centred
data matrix — no $400\times400$ covariance matrix needs to be formed.

In [ ]:
# ── 10.1 SVD of the centred SIMULATED expression matrix; uncentred for comparison ──
U_x, s_x, Vh_x = np.linalg.svd(Xc, full_matrices=False)              # thin SVD: U 24×24, s (24,), Vh 24×400
frac = s_x ** 2 / np.sum(s_x ** 2)
print("shapes: U", U_x.shape, "s", s_x.shape, "Vᵀ", Vh_x.shape)
print("singular values σ1..σ6:", s_x[:6].round(2))
print("fraction of variance  :", frac[:6].round(4), "  cumulative:", np.cumsum(frac)[:6].round(4))
print("smallest singular value:", f"{s_x[-1]:.2e}", "(centring removes one dimension: rank ≤ N − 1 = 23)")

pca_var = np.sort(np.linalg.eigvalsh(np.cov(Xc, rowvar=False)))[::-1][:3]
print("variances from the 400×400 covariance matrix:", pca_var.round(4), " σ²/(N−1):", (s_x[:3] ** 2 / 23).round(4))

s_raw = np.linalg.svd(X, compute_uv=False)
Vh_raw = np.linalg.svd(X, full_matrices=False)[2]
print(f"\nwithout centring: σ1 carries {s_raw[0] ** 2 / np.sum(s_raw ** 2):.1%} of Σσ², and v1 correlates "
      f"{abs(np.corrcoef(Vh_raw[0], X.mean(axis=0))[0, 1]):.4f} (in absolute value) with the mean expression of each gene")

🔍 **What just happened?** The thin SVD of the centred $24\times400$ matrix has 24 singular values; the first three,
$37.79, 37.09, 25.85$, explain $28.3\,\%$, $27.2\,\%$ and $13.2\,\%$ of the variance ($68.7\,\%$ together), and then the
values drop: components 4 and 5 (15.0, 14.5; about 4 % each) hold the *tissue-specific* parts of the infection response (see
below), and from the sixth on the values (9.8, 9.4, 9.2, …) form a slowly decaying noise floor. The last singular value is 0 up to rounding: centring
makes the 24 rows sum to zero, so the rank is at most 23. The variances computed the long way — eigenvalues of the
$400\times400$ covariance matrix — equal $\sigma_i^2/(N-1)$ (62.1, 59.8, 29.1). Without centring, the first singular vector
just reproduces the **average expression level** of each gene (correlation 1.0000 with the gene means, to four
decimals) and soaks up 99.1 % of $\sum\sigma^2$ — a reminder to centre before interpreting singular vectors as patterns of variation.

In [ ]:
# ── 10.2 Eigengenes (sample patterns u1, u2, u3) and the genes that drive the third pattern ──
scores = U_x[:, :3] * s_x[:3]                                        # PC scores = U Σ (one row per sample)
tissue_colour = {"spleen": "tab:purple", "kidney": "tab:green", "liver": "tab:brown"}
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
ax = axes[0]
ax.bar(range(1, 11), 100 * frac[:10], color="tab:blue")
ax.set_xlabel("component i"); ax.set_ylabel("% of variance (σ_i² / Σσ²)"); ax.set_title("scree plot")
ax = axes[1]
for (tissue, cond), sub in samples.groupby(["tissue", "condition"]):
    idx = sub.index.to_numpy()
    ax.scatter(scores[idx, 0], scores[idx, 1], color=tissue_colour[tissue], marker="o" if cond == "control" else "^",
               s=40, label=f"{tissue}, {cond}")
ax.set_xlabel("score on component 1 (σ1 u1)"); ax.set_ylabel("score on component 2 (σ2 u2)")
ax.set_title("components 1–2 separate the tissues"); ax.legend(fontsize=6.5)
ax = axes[2]
order = samples.sort_values(["condition", "tissue"]).index.to_numpy()
ax.bar(range(24), scores[order, 2], color=["tab:blue" if c == "control" else "tab:red"
                                             for c in samples.loc[order, "condition"]])
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks([5.5, 17.5]); ax.set_xticklabels(["12 control samples", "12 infected samples"])
ax.set_ylabel("score on component 3 (σ3 u3)"); ax.set_title("component 3 separates infection")
plt.tight_layout(); plt.show()

named = [g_ for gs in expr["pathways"].values() for g_ in gs]
top = np.argsort(-np.abs(Vh_x[2]))[:15]
print("15 genes with the largest |loading| on component 3 (v3):")
print(pd.DataFrame({"gene": genes[top], "loading": Vh_x[2, top].round(3),
                    "planted infection gene?": [g_ in named for g_ in genes[top]]}).to_string(index=False))
shared = set.intersection(*expr["true_de"].values())
print(f"\n🧪 ground truth: genes changed by infection in all three tissues = {sorted(shared)}")

🔍 **What just happened?** The scree plot shows two large components (about 28 % and 27 %), a third (13 %), two smaller ones (components 4 and 5, about 4 % each) and then
a tail. In the middle panel the samples fall into **three clusters by tissue** on components 1 and 2 — three groups
need two dimensions to be told apart, and the tissue differences (baseline levels of about a third of the genes) are the
largest source of variation. Within each tissue, controls (circles) and infected samples (triangles) form two small
sub-groups — infection leaves a trace on components 1 and 2 (and on 4 and 5, where the infected-minus-control shift has a different size
and sign in each tissue) through genes that respond in only some tissues — but none of those directions separates infection in
all tissues. Component 3 does: it is the **infection eigengene**: all 12 control samples have positive scores (about
3.2 to 7.6) and all 12 infected samples negative ones (about $-6.3$ to $-3.8$), in every tissue. (The sign of a
singular-vector pair is arbitrary — the separation is what matters.)

The gene loadings $\mathbf v_3$ (the eigenarray) tell us *which genes* carry the pattern: the 12 genes with the largest
$|$loading$|$ are exactly the 12 **planted infection-response genes** (SAA1, LYZ, IL6, MX1, TNF, CXCL8, IRF1, IL1B, C3,
STAT1, HAMP, ISG15), all with loadings of the same sign. The simulator's ground truth confirms that these 12 are precisely
the genes changed by infection in all three tissues. The next three, g280, g219 and g354, are genes the simulator made
respond in two tissues (spleen and kidney); g280 is *down*-regulated by infection, hence its loading of the opposite
sign. With
no labels at all, the SVD separated "which tissue" from "infected or not" and named the response programme.

### 🌉 Bridge — eigengenes, PCA and the genome-wide analyses that use them

Alter, Brown and Botstein (2000, *PNAS*) introduced exactly this analysis for genome-wide expression data: they called
the singular-vector patterns across arrays **eigengenes** and the patterns across genes **eigenarrays**, used the
fractions $\sigma_k^2/\sum\sigma^2$ to decide which patterns matter, and showed on yeast cell-cycle data that a few
eigengenes capture the steady state and the cell-cycle oscillations, while eigengenes that represent noise or
experimental artefacts can be filtered out by setting their singular values to zero — the truncation of Section 9. The
same SVD is the engine of PCA (scores $U\Sigma$, directions $V$), used to visualise single-cell data, detect batch effects
and population structure, and as the first step of almost every analysis of high-dimensional biology. The 02-620
companion derives PCA as variance maximisation and applies it to genotypes and images in
[BiologyToML · Module 14 — PCA](../../../../ml/tutorial/BiologyToML/notebooks/14_pca_dimensionality_reduction.ipynb), and to single-cell RNA-seq in
[Module 30](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb).

✏️ **Your turn.** (a) Repeat the SVD on the 8 **spleen** samples only. How many components are needed now, and which
one separates infection? (b) Build the rank-3 approximation $X_c^{(3)} = U_3\Sigma_3V_3^\top$ and compute the fraction of
$\|X_c\|_F^2$ it keeps; compare with the cumulative fraction printed in 10.1. (c) Standardise each gene (divide by its
standard deviation) before the SVD: do the components change? Why?

---
## 📌 Key takeaways

- **Similar matrices** $B' = P^{-1}BP$ describe the same linear map in another basis: same characteristic polynomial,
  eigenvalues, determinant, trace, rank. **Equivalent** matrices $A' = T^{-1}AS$ only share their rank. Similar ⇒ equivalent,
  not conversely.
- **Diagonalization.** $AP = PD$ ⇔ the columns of $P$ are eigenvectors and $D$ holds the eigenvalues (column by column).
  $A = X\Lambda X^{-1}$ exists iff $A$ has $n$ independent **eigenvectors** (guaranteed by $n$ distinct eigenvalues); the
  shear and the insect Leslie matrix (a repeated eigenvalue with too few eigenvectors) cannot be diagonalized.
- **Geometry.** $A = PDP^{-1}$: to eigen-coordinates, stretch, back. Symmetric $A = PDP^\top$: rotate, stretch, rotate back.
- **Powers.** $A^k = X\Lambda^kX^{-1}$ and $A^k\mathbf x = \sum_ic_i\lambda_i^k\mathbf x_i$: closed-form population projections,
  Markov chains converging to their stationary distribution, long-run behaviour set by the largest $|\lambda|$.
- **SVD.** Every $A\in\mathbb R^{m\times n}$ is $U\Sigma V^\top$ with orthogonal $U$, $V$ and a padded diagonal $\Sigma$,
  $\sigma_1\ge\sigma_2\ge\dots\ge0$; rank = number of non-zero $\sigma_i$; thin form: $U$ the shape of $A$, $\Sigma$ square.
  NumPy returns $V^\top$ (`Vh`: right-singular vectors in its **rows**).
- **Geometry of the SVD.** Rotate ($V^\top$), stretch and change dimension ($\Sigma$), rotate ($U$): the unit circle becomes an
  ellipse with semi-axes $\sigma_i\mathbf u_i$, and $A\mathbf v_i = \sigma_i\mathbf u_i$.
- **Construction.** $A^\top A = V\Sigma^\top\Sigma V^\top$: $\mathbf v_i$ = eigenvectors, $\sigma_i = \sqrt{\lambda_i}$,
  $\mathbf u_i = A\mathbf v_i/\sigma_i$. Worked example: $\begin{pmatrix}1&0&1\\-2&1&0\end{pmatrix}$ has $\sigma = \sqrt6, 1$.
- **Low rank.** Truncation is the best rank-$k$ approximation (Eckart–Young), error $\sum_{i>k}\sigma_i^2$: it denoises,
  predicts missing ratings and drug responses.
- **Biology.** SVD of the centred expression matrix = PCA: two components separate the tissues, the third separates
  infection, and its gene loadings name the 12 infection-response genes (eigengenes and eigenarrays).

---
## ✅ Checkpoint

1. Show that $\begin{pmatrix}1&2\\0&3\end{pmatrix}$ and $\begin{pmatrix}1&0\\0&2\end{pmatrix}$ are **not** similar.
   <details><summary>Answer</summary>Similar matrices have the same trace (and determinant). Traces 4 and 3 differ (determinants
   3 and 2 differ too), so no $P$ can exist.</details>
2. Why is every pair of similar matrices equivalent, and why are $I_2$ and $2I_2$ equivalent but not similar?
   <details><summary>Answer</summary>Take $S = T = P$ in $A' = T^{-1}AS$. $2I_2 = I_2^{-1}\,I_2\,(2I_2)$ shows equivalence ($T = I$,
   $S = 2I$); but $P^{-1}I_2P = I_2$ for every invertible $P$, so $2I_2$ is not similar to $I_2$ (eigenvalues 2, 2 vs 1, 1).</details>
3. Diagonalize $A = \begin{pmatrix}4&1\\2&3\end{pmatrix}$: give $X$, $\Lambda$, $X^{-1}$.
   <details><summary>Answer</summary>$\lambda^2 - 7\lambda + 10 = (\lambda-5)(\lambda-2)$; $\lambda = 5$: $(1,1)$; $\lambda = 2$: $(1,-2)$.
   $X = \begin{pmatrix}1&1\\1&-2\end{pmatrix}$, $\Lambda = \operatorname{diag}(5,2)$, $\det X = -3$,
   $X^{-1} = \frac{1}{3}\begin{pmatrix}2&1\\1&-1\end{pmatrix}$; check $X\Lambda X^{-1} = \frac13\begin{pmatrix}5&2\\5&-4\end{pmatrix}\begin{pmatrix}2&1\\1&-1\end{pmatrix}
   = \frac13\begin{pmatrix}12&3\\6&9\end{pmatrix} = A$.</details>
4. Is $\begin{pmatrix}2&0&0\\0&3&4\\0&0&5\end{pmatrix}$ diagonalizable? Is $\begin{pmatrix}2&1\\0&2\end{pmatrix}$?
   <details><summary>Answer</summary>The first is triangular with distinct diagonal entries 2, 3, 5 → three distinct eigenvalues →
   diagonalizable. The second has $\lambda = 2$ twice and $A - 2I = \begin{pmatrix}0&1\\0&0\end{pmatrix}$ has a one-dimensional
   null space → only one independent eigenvector → not diagonalizable.</details>
5. Using $A = \begin{pmatrix}1&2\\3&2\end{pmatrix} = X\Lambda X^{-1}$ with $\lambda = 4, -1$, find a formula for $A^k\mathbf u$,
   $\mathbf u = (1,2)$, and compute $A^3\mathbf u$.
   <details><summary>Answer</summary>$\mathbf u = 1.2(1, 1.5) - 0.05(4,-4)$, so $A^k\mathbf u = 1.2\cdot4^k(1,1.5) - 0.05(-1)^k(4,-4)$.
   $k = 3$: $76.8(1,1.5) + 0.05(4,-4) = (76.8 + 0.2,\ 115.2 - 0.2) = (77, 115)$.</details>
6. $A$ is $5\times3$. What are the shapes of $U$, $\Sigma$, $V^\top$ in the full SVD and in the thin SVD? How many singular
   values can be non-zero?
   <details><summary>Answer</summary>Full: $U$ $5\times5$, $\Sigma$ $5\times3$ (3 × 3 diagonal block + 2 zero rows), $V^\top$ $3\times3$. Thin: $U$
   $5\times3$, $\Sigma$ $3\times3$, $V^\top$ $3\times3$. At most $\min(5,3) = 3$ non-zero singular values (= the rank).</details>
7. Find the singular values of $A = \begin{pmatrix}3&0\\4&5\end{pmatrix}$.
   <details><summary>Answer</summary>$A^\top A = \begin{pmatrix}25&20\\20&25\end{pmatrix}$, eigenvalues $25\pm20 = 45, 5$, so
   $\sigma = \sqrt{45} = 3\sqrt5\approx6.71$ and $\sqrt5\approx2.24$. Check: $45 + 5 = 50 = 9 + 0 + 16 + 25$; $\sigma_1\sigma_2 = 15 = |\det A|$.</details>
8. Prove that $\|A\mathbf v_i\| = \sigma_i$ when $\mathbf v_i$ is a unit eigenvector of $A^\top A$ with eigenvalue $\lambda_i = \sigma_i^2$.
   <details><summary>Answer</summary>$\|A\mathbf v_i\|^2 = \mathbf v_i^\top A^\top A\mathbf v_i = \mathbf v_i^\top\lambda_i\mathbf v_i = \lambda_i\|\mathbf v_i\|^2 = \lambda_i$,
   so $\|A\mathbf v_i\| = \sqrt{\lambda_i} = \sigma_i$.</details>
9. A centred expression matrix has singular values 10, 6, 3, 1. What fraction of the variance do the first two components
   explain, and what is the Frobenius error of the best rank-2 approximation?
   <details><summary>Answer</summary>$\sum\sigma^2 = 100 + 36 + 9 + 1 = 146$; first two: $136/146\approx93.2\,\%$. Error
   $\sqrt{9 + 1} = \sqrt{10}\approx3.16$ (Eckart–Young).</details>
10. In `U, s, Vh = np.linalg.svd(Xc)` for a samples × genes matrix, which array holds the gene loadings of component 3,
    and which the sample scores?
    <details><summary>Answer</summary>Gene loadings: `Vh[2]` (the third **row** of $V^\top$, one entry per gene). Sample scores:
    `U[:, 2] * s[2]` (third column of $U$ times $\sigma_3$, one entry per sample).</details>
11. Why must the expression matrix be centred before its SVD is interpreted as PCA?
    <details><summary>Answer</summary>Only for centred data is $X_c^\top X_c/(N-1)$ the covariance matrix. Without centring, the first
    singular vector points to the mean expression profile (here 99.1 % of $\sum\sigma^2$, correlation ≈ 1.000 with the gene means)
    instead of a direction of variation.</details>

## 🧠 Questions & discussion

1. The SVD found "tissue" before "infection" because tissue differences carry more variance. Give an example where the
   biologically most interesting signal would sit in a **small** singular value — and how you would still find it.
2. A recommender predicts ratings from a low-rank model. What assumptions about people (or about drugs and cell lines)
   does "low rank" encode, and when would they fail?
3. Eigengenes are mathematical constructs: orthogonal, sorted by variance. Must a biological programme be orthogonal to
   every other programme? What could go wrong when you interpret $\mathbf u_2$ as "a pathway"?
4. Diagonalization makes $A^{100}$ easy, but only if $A$ is diagonalizable. In a real population model, would you worry
   about a repeated eigenvalue? (Think about how likely exact repeats are with measured vital rates.)
5. Missing values in a matrix may not be missing at random (a drug is not tested where it is expected to fail). How
   would that bias SVD-based completion?

## 🚀 Bonus — going further

- **Jordan normal form.** A non-diagonalizable matrix is similar to a block-diagonal matrix of Jordan blocks
  $\left(\begin{smallmatrix}\lambda&1\\0&\lambda\end{smallmatrix}\right)$. Use `sp.Matrix(...).jordan_form()` on the insect Leslie matrix of
  [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) and explain the factor $t$ in the transient $t(-1)^t$ of its projection.
- **Image compression.** Treat a grey-scale picture drawn by code (e.g. a simulated fluorescence image of nuclei) as a
  matrix and plot its rank-$k$ approximations for $k = 1, 5, 20$; how many numbers are stored?
- **Pseudo-inverse and least squares.** $A^+ = V\Sigma^+U^\top$ (invert the non-zero $\sigma_i$) solves least-squares problems
  even when $A^\top A$ is singular — compare `np.linalg.pinv` with the normal equations of [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb).
- **Randomised SVD.** For matrices with millions of rows (single-cell data), randomised algorithms compute the top $k$
  singular vectors fast; read about them in Halko, Martinsson & Tropp (2011, *SIAM Review*).
- **Gilbert Strang's lectures.** MIT OpenCourseWare 18.06 *Linear Algebra* (lectures on diagonalization and the SVD) and
  18.065 *Matrix Methods in Data Analysis* develop everything in this module further:
  [ocw.mit.edu](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 13 — Diagonalization and the singular value decomposition
Input: small typed-in matrices; an illustrative Leslie matrix; the background DNA Markov chain; a 6 × 5 ratings
       matrix; SIMULATED rank-3 data (30 × 200), drug responses (24 × 10) and the infection experiment (24 × 400)
Output: eigendecompositions X Λ X⁻¹, closed-form powers, SVD factors U Σ Vᵀ, low-rank approximations, eigengenes
Method: eigenvectors → X, eigenvalues → Λ; SVD from AᵀA (σ = √λ, u = Av/σ) or np.linalg.svd; truncation to rank k
Key equation: A = X Λ X⁻¹ (diagonalizable)   A = U Σ Vᵀ = Σ σ_i u_i v_iᵀ (always)   ‖A − A_k‖²_F = Σ_{i>k} σ_i²
Key parameters: rank k kept; centring/scaling of the data; number of imputation iterations
What changed when I modified … (k in 9.2/9.3, centring in 10.1, the tissue subset in 10.2):
One failure mode: (e.g. X singular for a non-diagonalizable matrix; reading Vh columns instead of rows; uncentred SVD)
Connection to 02-680: equivalent/similar matrices, diagonalization (AP = PD), eigendecomposition, the SVD theorem and
       shapes, SVD intuition (ℝ² → ℝ³), constructing the SVD (worked example), movie and gene-expression applications
```

## 📚 References

- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press —
  §4.4 (eigendecomposition and diagonalization), §4.5 (singular value decomposition, Example 4.13 and the
  $\mathbb R^2\to\mathbb R^3$ intuition), §4.6 (matrix approximation, Eckart–Young).
- Strang G. (2016) *Introduction to Linear Algebra*, 5th ed. Wellesley-Cambridge Press — §6.2 (diagonalizing a matrix),
  Chapter 7 (the singular value decomposition).
- Isaak S., Manougian M. N. (1976) *Basic Concepts of Linear Algebra*. W. W. Norton — similarity and diagonalization.
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — linear algebra chapters.
- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*, 2nd ed. Cambridge University Press.
- Alter O., Brown P. O., Botstein D. (2000) Singular value decomposition for genome-wide expression data processing and
  modeling. *Proceedings of the National Academy of Sciences USA* 97:10101–10106.
- Eckart C., Young G. (1936) The approximation of one matrix by another of lower rank. *Psychometrika* 1:211–218.
- Caswell H. (2001) *Matrix Population Models: Construction, Analysis, and Interpretation*, 2nd ed. Sinauer.
- Leslie P. H. (1945) On the use of matrices in certain population mathematics. *Biometrika* 33:183–212.
- Halko N., Martinsson P. G., Tropp J. A. (2011) Finding structure with randomness: probabilistic algorithms for
  constructing approximate matrix decompositions. *SIAM Review* 53:217–288.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.

## ➡️ Next

This module completes Part C — linear algebra. [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) opens Part D, **calculus**: instead of the fixed linear maps of
Modules 07–13, it studies how quantities **change** — growth rates of populations, elimination of drugs, the steepness
of a dose–response curve. Linear algebra returns there in a new role: in [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb) the eigenvalues of a Jacobian matrix
decide whether a biological equilibrium is stable, and the least-squares fit of a model to data rests on the matrix
$A^\top A$ you have just met.